# CEO Turnover and Executive Pay Dispersion in Europe
**Data & AI in Economics | TU Dortmund**

This notebook develops an analysis-ready panel for studying whether CEO turnover changes CEO compensation and pay dispersion in European firms. It begins with the STOXX Europe 600 universe, retrieves and validates BoardEx and Compustat data, identifies firm-level CEO spells and turnover events, matches CEOs to remuneration records, and combines the resulting CEO pay panel with firm fundamentals, market capitalization, executive characteristics, sector, and country information.

> **Team size:** 2 students  
> **Deliverable:** Jupyter Notebook, extended from proposal to final analysis


<a id="toc"></a>
## Table of Contents

1. [Team](#team)
2. [Research Question](#research-question)
3. [Data Sources and Variables](#data-sources-and-variables)
4. [Setup and Helper Functions](#helper-functions)
5. [Data Collection](#data-collection)
   - [STOXX 600 Universe](#stoxx-600-universe)
   - [BoardEx Employment Records](#boardex-employment-records)
   - [BoardEx Executive Details](#boardex-executive-details)
   - [BoardEx Remuneration Records](#boardex-remuneration-records)
   - [Compustat Fundamentals](#compustat-fundamentals)
6. [CEO Turnover and Pay Panel](#ceo-turnover-and-pay-panel)
   - [CEO Turnover Events](#ceo-turnover-events)
   - [CEO Remuneration Matching](#ceo-remuneration-matching)
   - [Firm-Year CEO Pay Panel](#firm-year-ceo-pay-panel)
7. [Market Data and Market Capitalization](#market-data-and-market-capitalization)
8. [Final Analysis Panel](#final-analysis-panel)
9. [Planned Methods](#planned-methods)
10. [Evaluation Strategy](#evaluation-strategy)
11. [Work Plan](#work-plan)
12. [Results and Discussion](#results-and-discussion)

**Reading guide:** run the notebook from top to bottom. The main dataset produced by the preparation workflow is `df_panel_final`; earlier tables are diagnostic checks that explain how each input contributes to that final panel.


<a id="team"></a>
## 1. Team


| Role | Name | Student ID |
|------|------|------------|
| Lead |Achmad Rizky Akbar| |
| Member | Kajetan Zduńczyk| |
| Member *(optional)* | | |


<a id="research-question"></a>
## 2. Mission Title & Research Question


**Title:** *CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes*

**Research question:** *Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

**Why it matters:** *By focusing on leadership changes observed in BoardEx, we can estimate how compensation responds to a governance shock without hand‑collecting policy data. This helps investors and regulators understand whether turnover acts as a disciplining mechanism on executive pay and internal pay gaps.*

<a id="data-sources-and-variables"></a>
## 3. Data Sources and Variables


**Source(s):**  

1. **BoardEx - Individual Profile Employment**

    Includes data about Individual Profile Employment for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-employment/

2. **BoardEx - Individual Profile Details**

    Includes data about Individual Profile Details for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-details/

3. **Company Profile Details - BoardEx (Wharton Data Research Services)**

    Company Profile Details includes data such as location, market cap, and sector.
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/company-profile/company-profile-details/

4. **Fundamentals Annual - Compustat Global (Wharton Data Research Services)**

    Provides fundamental annual company information
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/fundamentals-annual/

5. **Annual Remuneration - BoardEx (Wharton Data Research Services):**
    
    Data such as salary, bonus, and other cash compensation
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/compensation-analysis/annual-remuneration/

6. **Firms in Stoxx 600 Index**

    STOXX 600 is a major stock index representing the performance of 600 large-, mid-, and small-capitalization companies across 17 developed European countries.

    https://www.stoxx.com/selection-lists

**Table grains:** Employment data are role-spell records, remuneration data are executive-year records, fundamentals are firm-year records, and market prices are firm-security-month records. The final analysis table is reduced to one selected CEO observation per firm-year.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Total CEO compensation | Numeric | **Target** | Total annual compensation (salary + bonus + other cash/stock, in EUR) |
| Pay dispersion | Numeric | **Target / Outcome** | CEO pay relative to top-executive team (e.g., CEO-to-top-5 ratio) |
| CEO turnover indicator | Binary | **Treatment** | Flag for CEO change in a given year (from BoardEx role start/end) |
| Post‑turnover period | Binary | Feature | Indicator for years after turnover (event window) |
| Firm size (log assets / market cap) | Numeric | Feature | Scale and visibility of firm |
| Profitability (ROA / EBIT margin) | Numeric | Feature | Performance controls |
| Leverage | Numeric | Feature | Capital structure |
| Industry & country fixed effects | Categorical | Feature | Sector and institutional context |

**Planned outcomes and controls:**

The current preparation pipeline constructs CEO compensation and turnover variables. Pay dispersion, currency conversion, and inflation adjustment remain planned extensions and should be completed before the final modelling stage.

**Potential data quality issues:**  
- **Missing compensation components:** use multiple imputation or restrict to firms with complete pay breakdowns; report sensitivity to this choice.
- **Turnover date ambiguity:** define CEO change using role start/end dates and validate with overlapping roles; conduct robustness with alternative windows.
- **Reporting bias / top-coding:** winsorize extreme pay values; compare distributions by country to detect systematic reporting differences.
- **Selection bias in BoardEx coverage:** include a Stoxx 600 filter and check representativeness vs. population benchmarks.
- **Timing misalignment:** align fiscal-year fundamentals with compensation year; drop or lag inconsistent observations.
- **Currency and inflation effects:** convert to EUR and deflate using CPI to ensure comparability across years.

---

<a id="helper-functions"></a>
## 4. Setup and Helper Functions

This section defines reusable display, timing, connection, and caching utilities. Keeping these operations in one place makes the data pipeline easier to rerun and audit.


In [489]:
# Measure the running time of code blocks and print it in a readable format.

import time

def print_running_time(start_time, label="Running time"):
    """
    Print elapsed time since start_time.

    Parameters
    ----------
    start_time : float
        Start time from time.time()
    label : str
        Text label for the printed output
    """
    elapsed = time.time() - start_time

    if elapsed < 60:
        print(f"{label}: {elapsed:.2f} seconds")
    else:
        print(f"{label}: {elapsed:.2f} seconds ({elapsed / 60:.2f} minutes)")

In [490]:
# Show the number and percentage of missing values for each column in a DataFrame.

import pandas as pd

def missing_values_table(df, sort=True):
    """
    Show the number and percentage of missing values for each column.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to inspect.
    sort : bool
        If True, sort columns by missing percentage descending.

    Returns
    -------
    pandas.DataFrame
        Table with missing counts and percentages.
    """
    missing_count = df.isna().sum()
    missing_percent = (missing_count / len(df)) * 100

    missing_df = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    })

    missing_df["missing_percent"] = missing_df["missing_percent"].round(2)

    if sort:
        missing_df = missing_df.sort_values(
            by="missing_percent",
            ascending=False
        )

    return missing_df

In [491]:
# show all columns and rows in DataFrame outputs
pd.set_option('display.max_columns', None)

<a id="data-collection"></a>
## 5. Data Collection

The data pipeline follows a common STOXX 600 sample frame:

1. STOXX 600 ISINs define the firm universe.
2. BoardEx employment records provide CEO roles and turnover dates.
3. BoardEx executive details provide demographic characteristics.
4. BoardEx remuneration records provide annual compensation.
5. Compustat provides annual fundamentals and monthly market prices.

The sources are combined only after their identifiers, dates, and table grains have been checked.


<a id="stoxx-600-universe"></a>
### 5.1 STOXX 600 Company Universe


Before querying BoardEx, we define the project sample. The analysis is restricted to firms in the STOXX Europe 600. We use the `stoxx600_clean.csv` file as the sample frame and extract its ISINs as the main identifier for WRDS queries.

In [492]:
# Core packages and project paths
import wrds
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
STOXX_FILE = DATA_DIR / "stoxx600_clean.csv"
WRDS_CACHE_DIR = DATA_DIR / "wrds_cache"
WRDS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set this to True when you deliberately want to replace all cached WRDS data.
REFRESH_WRDS_CACHE = False

def get_wrds_connection():
    """Create the WRDS connection only when a server query is actually needed."""
    global conn

    try:
        conn
        print("Using existing WRDS connection.")
    except NameError:
        conn = wrds.Connection()
        print("Created new WRDS connection.")

    return conn


def load_or_fetch(cache_name, fetch_function, refresh=False):
    """Load a DataFrame from disk, or fetch and cache it when unavailable.

    Parameters
    ----------
    cache_name : str
        File stem used inside WRDS_CACHE_DIR.
    fetch_function : callable
        Zero-argument function that retrieves and returns a DataFrame.
    refresh : bool
        If True, ignore any existing cache and retrieve fresh data.
    """
    cache_path = WRDS_CACHE_DIR / f"{cache_name}.pkl"

    if cache_path.exists() and not refresh:
        df = pd.read_pickle(cache_path)
        print(f"Loaded cached data: {cache_path} ({len(df):,} rows)")
        return df

    print(f"Cache not used; retrieving data from WRDS: {cache_name}")
    df = fetch_function()

    if not isinstance(df, pd.DataFrame):
        raise TypeError("fetch_function must return a pandas DataFrame.")

    # Write to a temporary file first so an interrupted save cannot corrupt the cache.
    temporary_path = cache_path.with_suffix(".tmp")
    df.to_pickle(temporary_path)
    temporary_path.replace(cache_path)
    print(f"Saved data to cache: {cache_path} ({len(df):,} rows)")

    return df

print(f"STOXX 600 file exists: {STOXX_FILE.exists()}")
print(f"WRDS cache directory: {WRDS_CACHE_DIR.resolve()}")

STOXX 600 file exists: True
WRDS cache directory: /Users/acrizkyakbar/Documents/7. MASTER/2. SoSe 2026/Data and AI in Economics/DAI/mission/data/wrds_cache


WRDS results are cached as pickle files in `../data/wrds_cache`. On later runs, the notebook loads those local files and does not open a WRDS connection. A connection is created lazily only when a cache is missing or `REFRESH_WRDS_CACHE = True`.

In [493]:
# No connection is opened here. Each retrieval cell first checks its local cache.
print(f"Cached WRDS datasets available: {len(list(WRDS_CACHE_DIR.glob('*.pkl')))}")
print("A WRDS connection will be opened only if a required cache is unavailable.")

Cached WRDS datasets available: 6
A WRDS connection will be opened only if a required cache is unavailable.


Next, we read the STOXX 600 constituent file. The file contains 600 firms and includes ISIN, RIC, country, currency, exchange, and index membership information. The ISIN column is cleaned before it is passed to WRDS.

In [494]:
# Read STOXX Europe 600 constituents
df_sxxp = pd.read_csv(STOXX_FILE, sep=";")

df_sxxp["ISIN"] = (
    df_sxxp["ISIN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

isin_list = df_sxxp["ISIN"].dropna().unique().tolist()

print(f"STOXX 600 rows: {len(df_sxxp):,}")
print(f"Unique STOXX 600 ISINs: {len(isin_list):,}")
print(f"Duplicate ISIN rows: {df_sxxp['ISIN'].duplicated().sum():,}")

df_sxxp.head()

STOXX 600 rows: 600
Unique STOXX 600 ISINs: 600
Duplicate ISIN rows: 0


,Creation_Date,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership,Rank (FINAL)
0,20260501,546078,NL0010273215,ASML.AS,ASML HLDG,NL,EUR,Euronext Amsterdam,Large,1
1,20260501,40054,GB0005405286,HSBA.L,HSBC,GB,GBP,London SE,Large,2
2,20260501,98952,GB0009895292,AZN.L,ASTRAZENECA,GB,GBP,London SE,Large,3
3,20260501,474577,CH0012032048,ROPC.S,ROCHE PS,CH,CHF,Six Swiss Exchange,Large,4
4,20260501,477408,CH0012005267,NOVN.S,NOVARTIS,CH,CHF,Six Swiss Exchange,Large,5


**Table description:** The summary below checks the STOXX 600 sample frame before any WRDS joins. It confirms the number of unique ISINs, the country/currency mix, and whether duplicate securities appear in the index file.


In [495]:
display(df_sxxp.describe(include="all"))

,Creation_Date,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership,Rank (FINAL)
count,600.0,600,600,600,600,600,600,600,573,600.000000
unique,NaN,600,600,600,600,17,8,16,3,NaN
top,NaN,546078,NL0010273215,ASML.AS,ASML HLDG,GB,EUR,London SE,Large,NaN
freq,NaN,1,1,1,1,128,297,127,200,NaN
mean,20260501.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.500000
std,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,173.349358
min,20260501.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000
25%,20260501.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,150.750000
50%,20260501.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300.500000
75%,20260501.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450.250000


These ISINs are the bridge from the index universe to BoardEx. All following BoardEx employment and remuneration pulls should be filtered using this STOXX 600 firm list, so the project remains focused on European large- and mid-cap firms.

<a id="boardex-employment-records"></a>
### 5.2 BoardEx Employment Records


The STOXX 600 file defines the company universe for the project. We use the ISINs from `df_sxxp` as the filter for BoardEx Europe employment records, so all later CEO turnover and compensation variables are built only for firms that are members of the STOXX 600 sample.

This step pulls all employment records for people linked to the STOXX 600 firms. These records contain director identifiers, company identifiers, role names, role start dates, and role end dates. The role-date information is the basis for identifying CEO spells and later detecting CEO turnover events.

In [496]:
start_time = time.time()

# Pull BoardEx employment records for the STOXX 600 firm universe when not cached
isin_list = df_sxxp["ISIN"].dropna().str.strip().str.upper().unique().tolist()

def fetch_boardex_employment():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(isin_list), chunk_size):
        isin_chunk = tuple(isin_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT *
            FROM boardex.eur_wrds_dir_profile_emp
            WHERE isin IN %(isins)s
            ORDER BY companyname, directorname, datestartrole, rolename
        """, params={"isins": isin_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_emp = load_or_fetch(
    "boardex_employment_stoxx600",
    fetch_boardex_employment,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 ISINs used for WRDS query: {len(isin_list):,}")
print(f"BoardEx employment records retrieved: {len(df_emp):,}")
print(f"Unique companies matched in BoardEx: {df_emp['companyid'].nunique():,}")
print(f"Unique individuals matched in BoardEx: {df_emp['directorid'].nunique():,}")

print_running_time(start_time, label="Employment data load time")

df_emp.head()

Loaded cached data: ../data/wrds_cache/boardex_employment_stoxx600.pkl (141,814 rows)
STOXX 600 ISINs used for WRDS query: 600
BoardEx employment records retrieved: 141,814
Unique companies matched in BoardEx: 568
Unique individuals matched in BoardEx: 64,709
Employment data load time: 0.22 seconds


,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin
0,Listed Organisations,Aaron Church,3I GROUP PLC,2013-07-01,2022-04-28,No,Director - Infrastructure,UK,No,No,6366784.0,1348931.0,294.0,20.0,15.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
1,Listed Organisations,Aaron Church,3I GROUP PLC,2022-04-01,9000-01-01,No,Partner,Infrastructure United Kingdom,No,No,15137334.0,1348931.0,294.0,20.0,40.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
2,Listed Organisations,Adrian Yurkwich,3I GROUP PLC,1995-03-01,1998-10-28,No,Investment Executive,<NA>,No,No,7387811.0,1520966.0,294.0,20.0,15.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
3,Listed Organisations,Alain De Woillemont,3I GROUP PLC,2004-02-01,2007-09-28,No,Director - Development,Director Capital Development,No,No,8248610.0,1653996.0,294.0,20.0,15.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
4,Listed Organisations,Alan Lewis,3I GROUP PLC,1900-01-01,1991-12-28,No,Various Positions,<NA>,No,No,2548011.0,208004.0,294.0,75.0,25.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409


In [497]:
df_emp.describe(include="all")

,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin
count,141814,141814,141814,141814,141814,141814,141814,82302,141814,141814,141814.0,141814.0,141814.0,141814.0,141814.0,141814,141814,141814,141814
unique,1,64101,587,7244,6837,4,9972,64854,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,27,39,1,568
top,Listed Organisations,Doctor Roland Busch,DEUTSCHE BANK AG,1900-01-01,9000-01-01,No,Various Positions,Also Member of the Executive Committee,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,France,Banks,Quoted,DE0005140008
freq,141814,31,2725,10753,19347,112126,9387,767,119084,134014,<NA>,<NA>,<NA>,<NA>,<NA>,26800,27049,141814,2725
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9084735.393219,1387330.072701,248790.561059,23.384539,22.535751,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5482605.121622,920115.9061,675181.551884,16.713402,16.301246,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,37.0,294.0,10.0,10.0,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4491747.25,549660.0,9149.0,10.0,10.0,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8977588.5,1320084.5,22438.0,20.0,15.0,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14023242.75,2158757.25,30987.0,30.0,25.0,NaN,NaN,NaN,NaN


**Table description:** The missing-value table shows which employment fields are reliable enough for turnover construction. Role names, dates, `directorid`, `companyid`, and ISIN should have high coverage because they define CEO spells.


In [498]:
missing_values_table(df_emp)

,missing_count,missing_percent
fulltextdescription,59512,41.96
rowtype,0,0.00
primarykeyid,0,0.00
orgtype,0,0.00
sector,0,0.00
hocountryname,0,0.00
dateendroleflag,0,0.00
datestartroleflag,0,0.00
companyid,0,0.00
directorid,0,0.00


**Insights:**

- BoardEx matches slightly fewer firms than the 600 STOXX constituents, which is expected because coverage depends on BoardEx identifiers and available employment histories.
- The employment table is the backbone of the turnover design: it supplies `directorid`, `companyid`, role titles, and start/end dates.
- The most important quality check here is whether CEO-like role titles are too broad; later filters narrow regional or divisional CEO roles into firm-level CEO spells.


<a id="boardex-executive-details"></a>
### 5.3 BoardEx Executive Details

The employment table identifies which people belong to the sample. We use their `directorid` values to retrieve one executive-profile record per person, including age and gender for the final panel.


In [499]:
start_time = time.time()

# Retrieve executive details for all matched directors when not cached
directorid_list = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

def fetch_boardex_executive_details():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(directorid_list), chunk_size):
        directorid_chunk = tuple(directorid_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT *
            FROM boardex.eur_dir_profile_details
            WHERE directorid IN %(directorids)s
            ORDER BY directorname
        """, params={"directorids": directorid_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_exec = load_or_fetch(
    "boardex_executive_details_stoxx600",
    fetch_boardex_executive_details,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"Director IDs queried: {len(directorid_list):,}")
print(f"executives records retrieved: {len(df_exec):,}")
print(f"Unique executives: {df_exec['directorid'].nunique():,}")

print_running_time(start_time, label="Executive details load time")

df_exec

Loaded cached data: ../data/wrds_cache/boardex_executive_details_stoxx600.pkl (64,709 rows)
Director IDs queried: 64,709
executives records retrieved: 64,709
Unique executives: 64,709
Executive details load time: 0.05 seconds


,directorname,title,forename1,forename2,forename3,forename4,usualname,surname,suffixtitle,dob,dod,age,gender,nationality,recreations,directorvisible,linkedinurl,diversitynetworklabel,directorid,dobflag,dodflag,wealthxid,primaryroleid,networksize
0,Aaron Church,Mr,Aaron,T,<NA>,<NA>,<NA>,Church,<NA>,1900-01-01,9999-12-31,<NA>,M,<NA>,<NA>,Yes,https://www.linkedin.com/in/aaron-church-bb93452/,<NA>,1348931.0,75.0,55.0,4198568.0,13060661.0,2463.0
1,Admiral Rene Van Der Bruggen,Admiral,René,J,A,<NA>,<NA>,van der Bruggen,<NA>,1947-11-26,9999-12-31,76,M,Dutch,<NA>,Yes,<NA>,<NA>,327778.0,10.0,55.0,520642.0,4641608.0,367.0
2,Adriano Bandera,Mr,Adriano,<NA>,<NA>,<NA>,<NA>,Bandera,<NA>,1942-06-01,9999-12-31,83,M,Italian,<NA>,Yes,<NA>,<NA>,602849.0,10.0,55.0,<NA>,4378308.0,169.0
3,Adrian Yurkwich,Mr,Adrian,Michael,<NA>,<NA>,<NA>,Yurkwich,<NA>,1968-09-05,9999-12-31,57,M,British,<NA>,Yes,https://www.linkedin.com/in/adrian-yurkwich-27...,<NA>,1520966.0,10.0,55.0,<NA>,7382056.0,771.0
4,Adrienne Williams,Ms,Adrienne,<NA>,<NA>,<NA>,<NA>,Williams,<NA>,1900-01-01,9999-12-31,<NA>,F,<NA>,<NA>,Yes,https://www.linkedin.com/in/adrienne-williams-...,<NA>,2521149.0,75.0,55.0,<NA>,14471966.0,225.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64704,Yannick Hausmann,Mr,Yannick,Tim,<NA>,<NA>,<NA>,Hausmann,LLM,1967-09-01,9999-12-31,57,M,Swiss,<NA>,Yes,https://www.linkedin.com/in/hausmann-yannick-1...,<NA>,1074548.0,20.0,55.0,2231661.0,15384298.0,1930.0
64705,Yannis Kotziagkiaouridis,Mr,Yannis,<NA>,<NA>,<NA>,<NA>,Kotziagkiaouridis,<NA>,1900-01-01,9999-12-31,<NA>,M,<NA>,<NA>,Yes,https://www.linkedin.com/in/yanniskotziagkiaou...,<NA>,1334081.0,75.0,55.0,<NA>,16825237.0,696.0
64706,Yves Bonte,Mr,Yves,<NA>,<NA>,<NA>,<NA>,Bonte,<NA>,1961-01-01,9999-12-31,65,M,<NA>,<NA>,Yes,https://www.linkedin.com/in/yves-bonte-5a57975,<NA>,1055598.0,30.0,55.0,2344073.0,12737478.0,695.0
64707,Yvonne Jamal,Ms,Yvonne,<NA>,<NA>,<NA>,<NA>,Jamal,<NA>,1900-01-01,9999-12-31,<NA>,F,<NA>,<NA>,Yes,<NA>,<NA>,1541272.0,75.0,55.0,<NA>,12762448.0,57.0


<a id="boardex-remuneration-records"></a>
### 5.4 BoardEx Remuneration Records

We retrieve remuneration for all matched executives rather than CEOs alone. This supports the CEO-pay panel now and preserves the broader executive-pay population needed for the planned pay-dispersion outcome.


In [500]:
start_time = time.time()

# Retrieve remuneration for all executives when not cached (needed for pay dispersion)
directorid_list = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

def fetch_boardex_remuneration():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(directorid_list), chunk_size):
        directorid_chunk = tuple(directorid_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT
                boardname, boardid, directorname, directorid, rolename, rolestatus,
                annualreportdate, currency, salary, bonus, totalcompensation,
                valtoteqheld, valltipheld, valeqaward, ltipvalue, toteqatrisk,
                totremperiod, totaldirectcomp, perftotal
            FROM boardex.eur_dir_standard_remun
            WHERE directorid IN %(directorids)s
            ORDER BY boardname, directorname, annualreportdate
        """, params={"directorids": directorid_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_remun = load_or_fetch(
    "boardex_remuneration_stoxx600",
    fetch_boardex_remuneration,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"Director IDs queried: {len(directorid_list):,}")
print(f"Remuneration records retrieved: {len(df_remun):,}")
print(f"Unique remunerated directors: {df_remun['directorid'].nunique():,}")

print_running_time(start_time, label="Remuneration data load time")

df_remun.head()

Loaded cached data: ../data/wrds_cache/boardex_remuneration_stoxx600.pkl (276,820 rows)
Director IDs queried: 64,709
Remuneration records retrieved: 276,820
Unique remunerated directors: 26,114
Remuneration data load time: 0.12 seconds


,boardname,boardid,directorname,directorid,rolename,rolestatus,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,totremperiod,totaldirectcomp,perftotal
0,2G ENERGY AG,2037575.0,Pablo Hofelich,2228767.0,CEO,Pablo Hofelich has changed role on 12 Jun 2025,<NA>,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2INVEST AG (4basebio AG prior to 04/2021),18949.0,Juergen Dormann,11152.0,Chairman,Juergen Dormann departed 13 Oct 2004,2005-03-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,2024-03-01,USD,44.0,<NA>,44.0,<NA>,<NA>,<NA>,<NA>,<NA>,44.0,44.0,<NA>
3,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,2025-03-01,USD,67.0,<NA>,67.0,<NA>,<NA>,<NA>,<NA>,<NA>,67.0,67.0,<NA>
4,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,<NA>,USD,72.0,<NA>,72.0,<NA>,<NA>,<NA>,<NA>,<NA>,72.0,72.0,<NA>


**Table description:** The remuneration summary checks the scale and availability of CEO and executive pay variables. These fields are the source for salary, bonus, total compensation, equity-related compensation, and performance pay.


In [501]:
display(df_remun.describe(include="all"))

,boardname,boardid,directorname,directorid,rolename,rolestatus,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,totremperiod,totaldirectcomp,perftotal
count,276820,276820.0,276820,276820.0,276820,276820,260129,276820,48659.0,13119.0,67547.0,26747.0,7178.0,281.0,6890.0,8146.0,48632.0,67549.0,6890.0
unique,5920,<NA>,26064,<NA>,1998,106679,344,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
top,COMMERZBANK AG,<NA>,Baron Frère,<NA>,Independent Board Member,Franky Depickere joined this role on 15 Sep 2006,2022-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
freq,935,<NA>,189,<NA>,46193,36,13066,276820,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
mean,NaN,509344.892038,NaN,700384.020446,NaN,NaN,NaN,NaN,308.594464,759.050842,369.725672,55817.73698,7028.194065,788.551601,2200.292453,2242.198011,889.102484,443.623385,0.432226
std,NaN,908941.60544,NaN,749954.895803,NaN,NaN,NaN,NaN,531.924073,1024.877179,908.765732,1326127.049792,27747.836782,2848.260193,8243.914713,7855.340137,3679.636008,1154.123421,0.229422
min,NaN,304.0,NaN,37.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,NaN,12158.0,NaN,25033.0,NaN,NaN,NaN,NaN,54.0,82.0,0.0,38.0,621.0,80.0,247.0,232.0,64.0,0.0,0.26
50%,NaN,26246.0,NaN,488935.0,NaN,NaN,NaN,NaN,108.0,354.0,74.0,145.0,2120.0,188.0,943.5,930.0,127.0,79.0,0.42
75%,NaN,631694.0,NaN,1203298.0,NaN,NaN,NaN,NaN,283.0,1096.0,193.0,953.0,6041.75,323.0,2237.75,2287.0,352.0,220.0,0.59


In [502]:
missing_values_table(df_remun)

,missing_count,missing_percent
valeqaward,276539,99.90
perftotal,269930,97.51
ltipvalue,269930,97.51
valltipheld,269642,97.41
toteqatrisk,268674,97.06
bonus,263701,95.26
valtoteqheld,250073,90.34
totremperiod,228188,82.43
salary,228161,82.42
totalcompensation,209273,75.60


<a id="compustat-fundamentals"></a>
### 5.5 Compustat Annual Fundamentals

Compustat fundamentals add annual firm characteristics used as controls, including size, profitability, leverage, employment, and shares outstanding. The market-price section later uses `gvkey` to attach monthly prices near each fiscal reporting date.


In [503]:
start_time = time.time()

# Pull annual firm fundamentals for the STOXX 600 universe when not cached
isin_list = df_sxxp["ISIN"].dropna().str.strip().str.upper().unique().tolist()

def fetch_compustat_fundamentals():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(isin_list), chunk_size):
        isin_chunk = tuple(isin_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT  gvkey, isin, conm, fyear, datadate, loc, fic, curcd,
                    at, sale, emp, dltt, dlc, che, act, lct, wcap,
                    capx, xrd, aqc, nicon, ib, ceq, seq,
                    oiadp, oibdp, ebit, ebitda,
                    sich, naicsh, indfmt, exchg, cshoi
            FROM comp_global_daily.g_funda
            WHERE isin IN %(isins)s
              AND fyear BETWEEN 2000 AND 2024
            ORDER BY conm, fyear, datadate
        """, params={"isins": isin_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_firm = load_or_fetch(
    "compustat_fundamentals_stoxx600_2000_2024",
    fetch_compustat_fundamentals,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 ISINs used for WRDS query: {len(isin_list):,}")
print(f"Firm fundamentals records retrieved: {len(df_firm):,}")
print(f"Unique ISINs matched: {df_firm['isin'].nunique():,}")

print_running_time(start_time, label="Fundamentals data load time")

df_firm.head()

Loaded cached data: ../data/wrds_cache/compustat_fundamentals_stoxx600_2000_2024.pkl (12,501 rows)
STOXX 600 ISINs used for WRDS query: 600
Firm fundamentals records retrieved: 12,501
Unique ISINs matched: 556
Fundamentals data load time: 0.01 seconds


,gvkey,isin,conm,fyear,datadate,loc,fic,curcd,at,sale,emp,dltt,dlc,che,act,lct,wcap,capx,xrd,aqc,nicon,ib,ceq,seq,oiadp,oibdp,ebit,ebitda,sich,naicsh,indfmt,exchg,cshoi
0,210835,GB00B1YW4409,3I GROUP PLC,2000,2001-03-31,GBR,GBR,GBP,7439.0,<NA>,<NA>,1089.0,486.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-142.0,-142.0,4973.0,4973.0,-53.0,-27.0,-53.0,-27.0,6799,5239,FS,194,607.437
1,210835,GB00B1YW4409,3I GROUP PLC,2001,2002-03-31,GBR,GBR,GBP,6133.0,<NA>,<NA>,1271.0,154.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-960.0,-960.0,3945.0,3945.0,-823.0,-813.0,-823.0,-813.0,6799,5239,FS,194,599.887
2,210835,GB00B1YW4409,3I GROUP PLC,2002,2003-03-31,GBR,GBR,GBP,4999.0,<NA>,<NA>,1071.0,332.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,140.0,-935.0,2936.0,2936.0,-826.0,-819.0,-826.0,-819.0,6799,5239,FS,194,610.918
3,210835,GB00B1YW4409,3I GROUP PLC,2003,2004-03-31,GBR,GBR,GBP,5412.0,<NA>,0.75,1423.0,119.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,531.0,531.0,3395.0,3395.0,628.0,633.0,628.0,633.0,6799,5239,FS,194,603.594
4,210835,GB00B1YW4409,3I GROUP PLC,2004,2005-03-31,GBR,GBR,GBP,5701.0,<NA>,0.74,1312.0,206.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,512.0,512.0,3637.0,3637.0,597.0,601.0,597.0,601.0,6799,5239,FS,194,601.913


**Table description:** The fundamentals missing-value table identifies which Compustat variables are usable as firm controls. The final panel mainly relies on assets, sales, EBIT, long-term debt, net income, employees, shares outstanding, and market capitalization.


In [504]:
missing_values_table(df_firm)

,missing_count,missing_percent
xrd,6883,55.06
aqc,6185,49.48
capx,3366,26.93
wcap,3275,26.20
lct,3275,26.20
act,3275,26.20
che,3275,26.20
sale,3275,26.20
nicon,1178,9.42
emp,963,7.70


| Column | Description |
|---|---|
| `gvkey` | Global company key |
| `isin` | International Securities Identification Number |
| `conm` | Company name |
| `fyear` | Fiscal year |
| `datadate` | Fiscal-year reporting date |
| `loc` | Current ISO country code of headquarters |
| `fic` | Current ISO country code of incorporation |
| `curcd` | ISO currency code |
| `at` | Total assets |
| `sale` | Sales / turnover, net |
| `emp` | Number of employees |
| `dltt` | Long-term debt, total |
| `dlc` | Debt in current liabilities, total |
| `che` | Cash and short-term investments |
| `act` | Current assets, total |
| `lct` | Current liabilities, total |
| `wcap` | Working capital |
| `capx` | Capital expenditures |
| `xrd` | Research and development expense |
| `aqc` | Acquisitions |
| `nicon` | Net income/loss, consolidated |
| `ib` | Income before extraordinary items |
| `ceq` | Common/ordinary equity, total |
| `seq` | Stockholders’ equity, parent |
| `oiadp` | Operating income after depreciation |
| `oibdp` | Operating income before depreciation |
| `ebit` | Earnings before interest and taxes |
| `ebitda` | Earnings before interest, taxes, depreciation, and amortization |
| `sich` | Historical Standard Industrial Classification code |
| `naicsh` | Historical NAICS code |
| `indfmt` | Industry format |
| `exchg` | Stock exchange code |
| `cshoi` | Common shares outstanding |

---

<a id="ceo-turnover-and-pay-panel"></a>
## 6. CEO Turnover and Pay Panel

This section converts raw role and remuneration records into a firm-year CEO panel. The sequence is deliberate: identify valid CEO spells, detect changes between consecutive CEOs, match pay to the correct person and firm, and finally construct treatment and outcome variables.


<a id="ceo-turnover-events"></a>
### 6.1 Define CEO Turnover Events


#### 6.1.1 Identify broad CEO candidates from role names

In [506]:
# Identify broad CEO candidates from role names
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|chief executive officer|managing director"

# Clean dates before constructing CEO spells and turnover events
df_emp = df_emp.copy()
df_emp["datestartrole"] = pd.to_datetime(df_emp["datestartrole"], errors="coerce")

df_emp["dateendrole_clean"] = (
    df_emp["dateendrole"]
    .astype(str)
    .replace(["9000-01-01", "9999-12-31", "NaT", "nan", "None"], pd.NA)
)

df_emp["dateendrole_clean"] = pd.to_datetime(
    df_emp["dateendrole_clean"],
    errors="coerce"
)

df_emp["rolename_clean"] = df_emp["rolename"].fillna("").str.strip()

df_emp["is_ceo"] = (
    df_emp["rolename_clean"]
    .str.lower()
    .str.contains(CEO_PATTERN, na=False, regex=True)
)

df_emp_ceo = df_emp[df_emp["is_ceo"]].copy()

print(f"Broad CEO candidate role records: {len(df_emp_ceo):,}")
df_emp_ceo["rolename_clean"].value_counts().head(30)

Broad CEO candidate role records: 8,045


rolename_clean
Division CEO                       2688
CEO                                 725
Regional CEO                        600
President/CEO                       434
Chairman/CEO                        413
Division Chairman/Division CEO      257
Deputy CEO                          217
Division President/Division CEO     182
Division President/CEO              142
Division Deputy CEO                 123
Group CEO                           101
Division Chairman/CEO                99
Executive VP/Division CEO            97
Regional President/CEO               57
Division Chief Executive             47
Division Regional CEO                44
Division CEO/Division MD             44
CEO/MD                               39
Co-CEO                               38
CFO/Deputy CEO                       35
VP/Division CEO                      34
CEO/General Manager                  33
Division Interim CEO                 33
Regional CEO/Regional President      31
Executive VP/Regional CEO

#### 6.1.2 Refine candidates to firm-level CEO spells

In [507]:
# The broad CEO keyword search also captures regional, divisional, acting, and advisory roles.
df_emp_ceo = df_emp_ceo.copy()

# Main firm-level CEO keywords
has_main_ceo_title = df_emp_ceo["rolename_clean"].str.contains(
    r"\bCEO\b|Chief Executive|Chief Executive Officer|Group CEO|President/CEO|Chairman/CEO|Chair/CEO|MD/CEO|CEO/MD|Managing Director",
    case=False,
    regex=True,
    na=False
)

# Exclude business-unit / regional / lower-level CEO roles
exclude_unit_roles = df_emp_ceo["rolename_clean"].str.contains(
    r"Regional|Division|Divisional|Country|Branch|Sector|Zonal|Area|Business Unit|Market|Subsidiary|Segment",
    case=False,
    regex=True,
    na=False
)

# Exclude non-actual CEO roles
exclude_non_actual = df_emp_ceo["rolename_clean"].str.contains(
    r"Designate|Elect|in-Residence|Delegate|Office|Adviser|Advisor|Honorary|Shareholder Representative",
    case=False,
    regex=True,
    na=False
)

# Exclude subordinate CEO roles
exclude_subordinate = df_emp_ceo["rolename_clean"].str.contains(
    r"Deputy CEO|Vice CEO|Associate CEO|Assistant CEO",
    case=False,
    regex=True,
    na=False
)

# Interim / acting CEOs are excluded from the strict baseline definition
is_interim_acting = df_emp_ceo["rolename_clean"].str.contains(
    r"Interim|Acting",
    case=False,
    regex=True,
    na=False
)

# Co-CEO / joint CEO can be included in a robustness definition
is_co_ceo = df_emp_ceo["rolename_clean"].str.contains(
    r"Co-CEO|Joint CEO",
    case=False,
    regex=True,
    na=False
)

# Baseline CEO role definition
df_emp_ceo["firm_level_ceo_role"] = (
    has_main_ceo_title
    & ~exclude_unit_roles
    & ~exclude_non_actual
    & ~exclude_subordinate
)

# Strict baseline: exclude interim/acting and co-CEO roles
df_emp_ceo["firm_level_ceo_strict"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
    & ~is_co_ceo
)

# Robustness definition: include co-CEOs but exclude interim/acting CEOs
df_emp_ceo["firm_level_ceo_with_coceo"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
)

print(f"Strict firm-level CEO spells: {df_emp_ceo['firm_level_ceo_strict'].sum():,}")
print(f"Firm-level CEO spells including co-CEOs: {df_emp_ceo['firm_level_ceo_with_coceo'].sum():,}")

df_emp_ceo

Strict firm-level CEO spells: 2,039
Firm-level CEO spells including co-CEOs: 2,114


,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin,dateendrole_clean,rolename_clean,is_ceo,firm_level_ceo_role,firm_level_ceo_strict,firm_level_ceo_with_coceo
196,Listed Organisations,Maite Ballester Fornés,3I GROUP PLC,2008-01-01,2014-03-28,No,Regional CEO,Also Partner and MD Spain,No,No,5895606.0,1297838.0,294.0,20.0,15.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409,2014-03-28,Regional CEO,True,False,False,False
301,Listed Organisations,Simon Borrows,3I GROUP PLC,2012-05-17,9000-01-01,Yes,CEO,Also Chairman of the Executive Committee,No,No,5422761.0,342366.0,294.0,10.0,40.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409,NaT,CEO,True,True,True,True
392,Listed Organisations,Edoardo Iacopozzi,A2A SPA,2024-01-01,9000-01-01,No,Division CEO,Services & Real Estate,No,No,17925979.0,2482408.0,953.0,30.0,40.0,Italy,Utilities - Other,Quoted,IT0001233417,NaT,Division CEO,True,False,False,False
454,Listed Organisations,Luca Camerano,A2A SPA,2014-06-16,2017-07-11,Yes,CEO,<NA>,No,No,7688969.0,1587233.0,953.0,10.0,10.0,Italy,Utilities - Other,Quoted,IT0001233417,2017-07-11,CEO,True,True,True,True
455,Listed Organisations,Luca Camerano,A2A SPA,2017-07-11,2020-05-13,Yes,CEO/General Manager,<NA>,No,No,10084148.0,1587233.0,953.0,10.0,10.0,Italy,Utilities - Other,Quoted,IT0001233417,2020-05-13,CEO/General Manager,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141784,Listed Organisations,Tulsi Naidu,ZURICH INSURANCE GROUP AG,2021-01-01,9000-01-01,No,Regional CEO,Asia-Pacific Also Member of the Executive Comm...,No,Yes,13357074.0,1286256.0,34200.0,10.0,40.0,Switzerland,Insurance,Quoted,CH0011075394,NaT,Regional CEO,True,False,False,False
141790,Listed Organisations,Urs Lüthy,ZURICH INSURANCE GROUP AG,2026-03-10,9000-01-01,No,Division CEO,Also Head Commercial Insurance Zurich Switzerland,No,No,18423134.0,3257780.0,34200.0,10.0,40.0,Switzerland,Insurance,Quoted,CH0011075394,NaT,Division CEO,True,False,False,False
141798,Listed Organisations,Wendy Liu,ZURICH INSURANCE GROUP AG,2021-09-01,2022-04-28,No,Division Interim CEO,Zurich International,No,No,15830577.0,1317310.0,34200.0,20.0,15.0,Switzerland,Insurance,Quoted,CH0011075394,2022-04-28,Division Interim CEO,True,False,False,False
141799,Listed Organisations,Wendy Liu,ZURICH INSURANCE GROUP AG,2022-05-01,9000-01-01,No,Division CEO,Zurich Integrated Benefits and International Life,No,No,15830580.0,1317310.0,34200.0,20.0,40.0,Switzerland,Insurance,Quoted,CH0011075394,NaT,Division CEO,True,False,False,False


#### 6.1.3 Construct the baseline CEO-spell table

In [508]:
df_ceo_main = (
    df_emp_ceo[df_emp_ceo["firm_level_ceo_strict"]]
    .dropna(subset=["companyid", "directorid", "datestartrole"])
    .sort_values(["companyid", "datestartrole", "dateendrole_clean", "directorid"])
    .copy()
)

df_ceo_main

,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin,dateendrole_clean,rolename_clean,is_ceo,firm_level_ceo_role,firm_level_ceo_strict,firm_level_ceo_with_coceo
301,Listed Organisations,Simon Borrows,3I GROUP PLC,2012-05-17,9000-01-01,Yes,CEO,Also Chairman of the Executive Committee,No,No,5422761.0,342366.0,294.0,10.0,40.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409,NaT,CEO,True,True,True,True
743,Listed Organisations,Jan Aalberts,AALBERTS NV (Aalberts Industries NV prior to 0...,1987-01-01,2012-04-26,Yes,President/CEO,Also Founder,No,No,874322.0,33284.0,384.0,30.0,10.0,Netherlands,Engineering & Machinery,Quoted,NL0000852564,2012-04-26,President/CEO,True,True,True,True
795,Listed Organisations,Wim Pelsma,AALBERTS NV (Aalberts Industries NV prior to 0...,2012-04-26,2023-09-07,Yes,CEO,<NA>,No,No,5056177.0,654106.0,384.0,10.0,10.0,Netherlands,Engineering & Machinery,Quoted,NL0000852564,2023-09-07,CEO,True,True,True,True
785,Listed Organisations,Stéphane Simonetta,AALBERTS NV (Aalberts Industries NV prior to 0...,2023-09-07,9000-01-01,Yes,CEO,<NA>,No,No,15951081.0,1542238.0,384.0,10.0,40.0,Netherlands,Engineering & Machinery,Quoted,NL0000852564,NaT,CEO,True,True,True,True
1092,Listed Organisations,Jörgen Centerman,ABB LTD,2000-12-31,2002-09-05,Yes,President/CEO,<NA>,No,No,16639.0,14884.0,422.0,10.0,10.0,Switzerland,Engineering & Machinery,Quoted,CH0012221716,2002-09-05,President/CEO,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95311,Listed Organisations,Marc Puig Guasch,PUIG BRANDS SA,2024-05-03,9000-01-01,Yes,Chairman/CEO,<NA>,No,No,16722729.0,1680890.0,3786106.0,10.0,40.0,Spain,Clothing & Personal Products,Quoted,ES0105777017,NaT,Chairman/CEO,True,True,True,True
117751,Listed Organisations,André Krause,SUNRISE COMMUNICATIONS AG,2024-11-08,9000-01-01,No,CEO,Principal Executive Officer,No,Yes,17208226.0,1109005.0,3883566.0,10.0,40.0,Switzerland,Telecommunication Services,Quoted,CH1386220409,NaT,CEO,True,True,True,True
136422,Listed Organisations,Austin Lally,VERISURE PLC,2025-10-08,9000-01-01,Yes,CEO,<NA>,No,No,18084317.0,1262461.0,4030700.0,10.0,40.0,United Kingdom - England,Business Services,Quoted,GB00BVMN1558,NaT,CEO,True,True,True,True
77449,Listed Organisations,Peter ter Kulve,MAGNUM ICE CREAM CO NV (THE) (TMICC),2025-12-08,9000-01-01,Yes,CEO,<NA>,No,No,18223838.0,1342012.0,4052924.0,10.0,40.0,Netherlands,Food Producers & Processors,Quoted,NL0015002MS2,NaT,CEO,True,True,True,True


#### 6.1.4 Identify CEO turnover events

CEO turnover is defined when the current firm-level CEO's `directorid` differs from the previous observed CEO's `directorid` within the same company. The first observed CEO spell per company is not coded as turnover, because we do not observe a prior CEO inside our sample window.

In [509]:
# Using .ne(...).fillna(False) avoids pandas NA-to-integer conversion errors.
df_ceo_main["previous_ceo_directorid"] = (
    df_ceo_main
    .groupby("companyid")["directorid"]
    .shift()
)

df_ceo_main["ceo_turnover"] = (
    df_ceo_main["directorid"]
    .ne(df_ceo_main["previous_ceo_directorid"])
    .fillna(False)
    .astype(int)
)

# First observed CEO in each company is not counted as turnover
df_ceo_main.loc[
    df_ceo_main["previous_ceo_directorid"].isna(),
    "ceo_turnover"
] = 0

df_ceo_main["turnover_year"] = df_ceo_main["datestartrole"].dt.year

print(f"CEO role spells: {len(df_ceo_main):,}")
print(f"Unique companies with CEO spells: {df_ceo_main['companyid'].nunique():,}")
print(f"CEO turnover events: {df_ceo_main['ceo_turnover'].sum():,}")

df_ceo_main[[
    "companyname", "directorname", "rolename", "datestartrole",
    "dateendrole_clean", "previous_ceo_directorid", "ceo_turnover", "turnover_year"
]].head(10)

CEO role spells: 2,039
Unique companies with CEO spells: 513
CEO turnover events: 1,238


,companyname,directorname,rolename,datestartrole,dateendrole_clean,previous_ceo_directorid,ceo_turnover,turnover_year
301,3I GROUP PLC,Simon Borrows,CEO,2012-05-17,NaT,<NA>,0,2012
743,AALBERTS NV (Aalberts Industries NV prior to 0...,Jan Aalberts,President/CEO,1987-01-01,2012-04-26,<NA>,0,1987
795,AALBERTS NV (Aalberts Industries NV prior to 0...,Wim Pelsma,CEO,2012-04-26,2023-09-07,33284.0,1,2012
785,AALBERTS NV (Aalberts Industries NV prior to 0...,Stéphane Simonetta,CEO,2023-09-07,NaT,654106.0,1,2023
1092,ABB LTD,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,<NA>,0,2000
1103,ABB LTD,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,14884.0,1,2002
1017,ABB LTD,Fred Kindle,President/CEO,2005-01-01,2008-02-13,11152.0,1,2005
973,ABB LTD,Doctor Ulrich Spiesshofer,President/CEO,2013-09-15,2019-04-17,4611.0,1,2013
853,ABB LTD,Björn Rosengren,CEO,2020-03-01,2024-07-31,486454.0,1,2020
1194,ABB LTD,Morten Wierod,CEO,2024-08-01,NaT,32417.0,1,2024


#### 6.1.5 Collapse turnover events to the firm-year level

In [510]:
ceo_turnover_firm_year = (
    df_ceo_main[df_ceo_main["ceo_turnover"] == 1]
    .groupby(["companyid", "turnover_year"])
    .size()
    .reset_index(name="num_ceo_turnovers")
)

ceo_turnover_firm_year["ceo_turnover_dummy"] = 1
ceo_turnover_firm_year.head(20)

,companyid,turnover_year,num_ceo_turnovers,ceo_turnover_dummy
0,384.0,2012,1,1
1,384.0,2023,1,1
2,422.0,2002,1,1
3,422.0,2005,1,1
4,422.0,2013,1,1
5,422.0,2020,1,1
6,422.0,2024,1,1
7,595.0,2016,1,1
8,595.0,2021,1,1
9,598.0,2006,1,1


The resulting `ceo_turnover_firm_year` table will later be merged into the remuneration panel by `companyid` and fiscal/report year.

<a id="ceo-remuneration-matching"></a>
### 6.2 Prepare Annual Remuneration Records for Identified CEOs


The cached remuneration table contains all executives in the sampled firms. We now filter it to the `directorid` values identified as firm-level CEOs, clean the compensation variables, and then match each record to the correct CEO spell and company.

In [511]:
df_remun

,boardname,boardid,directorname,directorid,rolename,rolestatus,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,totremperiod,totaldirectcomp,perftotal
0,2G ENERGY AG,2037575.0,Pablo Hofelich,2228767.0,CEO,Pablo Hofelich has changed role on 12 Jun 2025,<NA>,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2INVEST AG (4basebio AG prior to 04/2021),18949.0,Juergen Dormann,11152.0,Chairman,Juergen Dormann departed 13 Oct 2004,2005-03-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,2024-03-01,USD,44.0,<NA>,44.0,<NA>,<NA>,<NA>,<NA>,<NA>,44.0,44.0,<NA>
3,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,2025-03-01,USD,67.0,<NA>,67.0,<NA>,<NA>,<NA>,<NA>,<NA>,67.0,67.0,<NA>
4,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,<NA>,USD,72.0,<NA>,72.0,<NA>,<NA>,<NA>,<NA>,<NA>,72.0,72.0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276815,ZURICH INSURANCE GROUP AG,34200.0,Rafael del Pino y Calvo-Sotelo,27757.0,Independent Director,Rafael del Pino y Calvo-Sotelo joined this rol...,2014-12-01,USD,243.0,<NA>,243.0,292.0,<NA>,<NA>,<NA>,<NA>,243.0,293.0,<NA>
276816,ZURICH INSURANCE GROUP AG,34200.0,Rafael del Pino y Calvo-Sotelo,27757.0,Independent Director,Rafael del Pino y Calvo-Sotelo joined this rol...,2015-12-01,USD,132.0,<NA>,132.0,351.0,<NA>,<NA>,<NA>,<NA>,132.0,190.0,<NA>
276817,ZURICH INSURANCE GROUP AG,34200.0,Rafael del Pino y Calvo-Sotelo,27757.0,Independent Director,Rafael del Pino y Calvo-Sotelo departed 30 Mar...,2016-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
276818,ZURICH INSURANCE GROUP AG,34200.0,Rolf Hüppi,8796.0,Chairman/CEO,Rolf Hüppi joined this role in Oct 2000,2001-12-01,USD,<NA>,<NA>,0.0,7634.0,<NA>,<NA>,920.0,920.0,920.0,0.0,1.0


#### 6.2.1 Filter remuneration to identified CEOs

In [512]:
ceo_directorid_list = (
    df_ceo_main["directorid"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

df_remun_ceos = df_remun[df_remun["directorid"].isin(ceo_directorid_list)].copy()

print(f"CEO director IDs queried: {len(ceo_directorid_list):,}")
print(f"Remuneration records retrieved: {len(df_remun_ceos):,}")
print(f"Unique remunerated directors: {df_remun_ceos['directorid'].nunique():,}")
print(f"Remuneration records for CEOs: {len(df_remun_ceos):,}")

df_remun_ceos

CEO director IDs queried: 1,653
Remuneration records retrieved: 31,796
Unique remunerated directors: 1,449
Remuneration records for CEOs: 31,796


,boardname,boardid,directorname,directorid,rolename,rolestatus,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,totremperiod,totaldirectcomp,perftotal
1,2INVEST AG (4basebio AG prior to 04/2021),18949.0,Juergen Dormann,11152.0,Chairman,Juergen Dormann departed 13 Oct 2004,2005-03-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
150,A2A SPA,953.0,Flavio Cattaneo,9246.0,Deputy Chairman,Flavio Cattaneo joined this role in 1999,1999-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
151,A2A SPA,953.0,Flavio Cattaneo,9246.0,Deputy Chairman,Flavio Cattaneo joined this role in 1999,2000-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
152,A2A SPA,953.0,Flavio Cattaneo,9246.0,Deputy Chairman,Flavio Cattaneo departed 29 Jan 2001,2001-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
247,A2A SPA,953.0,Luca Camerano,1587233.0,CEO,Luca Camerano joined this role on 16 Jun 2014,2014-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276706,ZEALAND PHARMA AS,45895.0,Britt Jensen,1477956.0,President/CEO,Britt Jensen joined this role on 15 Jan 2015,2017-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
276707,ZEALAND PHARMA AS,45895.0,Britt Jensen,1477956.0,President/CEO,Britt Jensen joined this role on 15 Jan 2015,2018-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
276708,ZEALAND PHARMA AS,45895.0,Britt Jensen,1477956.0,President/CEO,Britt Jensen departed 28 Feb 2019,2019-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
276818,ZURICH INSURANCE GROUP AG,34200.0,Rolf Hüppi,8796.0,Chairman/CEO,Rolf Hüppi joined this role in Oct 2000,2001-12-01,USD,<NA>,<NA>,0.0,7634.0,<NA>,<NA>,920.0,920.0,920.0,0.0,1.0


We clean the remuneration date and compensation variables before merging. `annualreportdate` identifies the financial reporting year for the compensation observation. `totalcompensation` is the main CEO pay outcome for the first version of the analysis.

#### 6.2.2 Clean remuneration dates and compensation variables

In [513]:
df_remun_ceos = df_remun_ceos.copy()

df_remun_ceos["annualreportdate"] = pd.to_datetime(
    df_remun_ceos["annualreportdate"],
    errors="coerce"
)

df_remun_ceos["annualreportyear"] = df_remun_ceos["annualreportdate"].dt.year

pay_cols = [
    "salary", "bonus", "totalcompensation", "valtoteqheld", "valltipheld",
    "valeqaward", "ltipvalue", "toteqatrisk", "totremperiod",
    "totaldirectcomp", "perftotal"
]

for col in pay_cols:
    if col in df_remun_ceos.columns:
        df_remun_ceos[col] = pd.to_numeric(df_remun_ceos[col], errors="coerce")

print("Missing annual report dates:", df_remun_ceos["annualreportdate"].isna().sum())
print("Missing total compensation:", df_remun_ceos["totalcompensation"].isna().sum())
print("Positive total compensation records:", (df_remun_ceos["totalcompensation"] > 0).sum())

df_remun_ceos[["directorid", "directorname", "boardid", "boardname", "annualreportdate", "currency", "totalcompensation"]]

Missing annual report dates: 1488
Missing total compensation: 21611
Positive total compensation records: 8054


,directorid,directorname,boardid,boardname,annualreportdate,currency,totalcompensation
1,11152.0,Juergen Dormann,18949.0,2INVEST AG (4basebio AG prior to 04/2021),2005-03-01,USD,<NA>
150,9246.0,Flavio Cattaneo,953.0,A2A SPA,1999-12-01,USD,<NA>
151,9246.0,Flavio Cattaneo,953.0,A2A SPA,2000-12-01,USD,<NA>
152,9246.0,Flavio Cattaneo,953.0,A2A SPA,2001-12-01,USD,<NA>
247,1587233.0,Luca Camerano,953.0,A2A SPA,2014-12-01,USD,<NA>
...,...,...,...,...,...,...,...
276706,1477956.0,Britt Jensen,45895.0,ZEALAND PHARMA AS,2017-12-01,USD,<NA>
276707,1477956.0,Britt Jensen,45895.0,ZEALAND PHARMA AS,2018-12-01,USD,<NA>
276708,1477956.0,Britt Jensen,45895.0,ZEALAND PHARMA AS,2019-12-01,USD,<NA>
276818,8796.0,Rolf Hüppi,34200.0,ZURICH INSURANCE GROUP AG,2001-12-01,USD,0.0


### 6.3 Match CEO Remuneration to Valid CEO Spells


The merge first links remuneration and employment by `directorid`, then requires `boardid == companyid` so compensation cannot be assigned to the same person at another firm. We retain annual-report dates inside the CEO spell with a 90-day boundary tolerance. The tolerance accommodates reporting dates close to a transition, but it can allow both outgoing and incoming CEOs to appear in the same firm-year; the next section therefore applies an explicit one-record selection rule.

In [514]:
ceo_spells = df_ceo_main[[
    "companyid", "companyname", "directorid", "directorname", "rolename",
    "datestartrole", "dateendrole_clean", "ceo_turnover", "turnover_year"
]].copy()

# Ensure merge keys are comparable
ceo_spells["companyid"] = pd.to_numeric(ceo_spells["companyid"], errors="coerce")
df_remun_ceos["boardid"] = pd.to_numeric(df_remun_ceos["boardid"], errors="coerce")


# df_rt_ceo = remuneration records for CEO candidates matched to valid CEO spells
df_rt_ceos = df_remun_ceos.merge(
    ceo_spells,
    on="directorid",
    how="inner",
    suffixes=("_remun", "_emp")
)

# Keep compensation records for the same firm only
df_rt_ceos = df_rt_ceos[
    df_rt_ceos["boardid"].eq(df_rt_ceos["companyid"])
].copy()

# Keep compensation records during the valid CEO role period
tolerance_days = 90
start_ok = df_rt_ceos["annualreportdate"] >= (
    df_rt_ceos["datestartrole"] - pd.Timedelta(days=tolerance_days)
)
end_ok = (
    df_rt_ceos["dateendrole_clean"].isna()
    | (df_rt_ceos["annualreportdate"] <= df_rt_ceos["dateendrole_clean"] + pd.Timedelta(days=tolerance_days))
)

df_rt_ceos = df_rt_ceos[start_ok & end_ok].copy()

# Keep usable compensation observations for the baseline CEO pay analysis
df_rt_ceos = df_rt_ceos[
    df_rt_ceos["annualreportdate"].notna()
    & df_rt_ceos["totalcompensation"].notna()
    & (df_rt_ceos["totalcompensation"] > 0) # non-zero total compensation
].drop_duplicates().copy()

df_rt_ceos["annualreportyear"] = df_rt_ceos["annualreportdate"].dt.year
df_rt_ceos["log_totalcompensation"] = np.log(df_rt_ceos["totalcompensation"])

print(f"Matched CEO remuneration records: {len(df_rt_ceos):,}")
print(f"Companies with matched CEO pay: {df_rt_ceos['companyid'].nunique():,}")
print(f"CEOs with matched pay: {df_rt_ceos['directorid'].nunique():,}")

df_rt_ceos[[
    "companyname", "directorname_emp", "rolename_emp", "datestartrole",
    "dateendrole_clean", "annualreportdate", "currency", "totalcompensation",
    "log_totalcompensation", "ceo_turnover", "turnover_year"
]].head(10)

Matched CEO remuneration records: 2,119
Companies with matched CEO pay: 138
CEOs with matched pay: 355


,companyname,directorname_emp,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,currency,totalcompensation,log_totalcompensation,ceo_turnover,turnover_year
158,ABB LTD,Doctor Ulrich Spiesshofer,President/CEO,2013-09-15,2019-04-17,2013-12-01,USD,2742.0,7.916443,1,2013
159,ABB LTD,Doctor Ulrich Spiesshofer,President/CEO,2013-09-15,2019-04-17,2014-12-01,USD,3701.0,8.216358,1,2013
160,ABB LTD,Doctor Ulrich Spiesshofer,President/CEO,2013-09-15,2019-04-17,2015-12-01,USD,4192.0,8.340933,1,2013
161,ABB LTD,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2004-12-01,USD,351.0,5.860786,1,2005
164,ABB LTD,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,USD,1299.0,7.16935,1,2005
167,ABB LTD,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2006-12-01,USD,2689.0,7.896925,1,2005
170,ABB LTD,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2007-12-01,USD,3059.0,8.025843,1,2005
229,ABB LTD,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,USD,1787.0,7.488294,0,2000
238,ABB LTD,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,USD,1419.0,7.257708,1,2002
241,ABB LTD,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,USD,2414.0,7.78904,1,2002


<a id="firm-year-ceo-pay-panel"></a>
### 6.4 Firm-Year CEO Pay Panel


The matched table can contain more than one CEO record for a company-year when outgoing and incoming CEOs both receive compensation, when roles overlap, or when the 90-day tolerance admits records near both spells. For this first analysis panel, we keep the latest annual-report observation within each `companyid` and `annualreportyear`. If annual-report dates tie, the current sort uses `directorid` only as a deterministic tie-breaker; that tie should not be interpreted economically.

In [515]:
# Build a firm-year CEO pay panel from the matched remuneration records
panel_cols = [
    "companyid", "companyname", "boardid", "boardname",
    "directorid", "directorname_emp", "rolename_emp",
    "datestartrole", "dateendrole_clean",
    "annualreportdate", "annualreportyear", "currency",
    "salary", "bonus", "totalcompensation", "log_totalcompensation",
    "totaldirectcomp", "perftotal", "valtoteqheld", "valltipheld",
    "valeqaward", "ltipvalue", "toteqatrisk",
    "ceo_turnover", "turnover_year"
]

available_panel_cols = [col for col in panel_cols if col in df_rt_ceos.columns]
missing_panel_cols = [col for col in panel_cols if col not in df_rt_ceos.columns]

if missing_panel_cols:
    print(f"Panel columns unavailable and omitted: {missing_panel_cols}")

ceo_pay_panel = df_rt_ceos[available_panel_cols].copy()

# Keep one record per firm-year: latest report date, then directorid as a deterministic tie-breaker.
ceo_pay_panel = (
    ceo_pay_panel
    .sort_values(["companyid", "annualreportyear", "annualreportdate", "directorid"])
    .drop_duplicates(subset=["companyid", "annualreportyear"], keep="last")
    .sort_values(["companyid", "annualreportyear"])
    .reset_index(drop=True)
)

print(f"Firm-year CEO pay observations: {len(ceo_pay_panel):,}")
print(f"Unique companies: {ceo_pay_panel['companyid'].nunique():,}")
print(f"Year range: {ceo_pay_panel['annualreportyear'].min()}-{ceo_pay_panel['annualreportyear'].max()}")

ceo_pay_panel

Firm-year CEO pay observations: 2,069
Unique companies: 138
Year range: 1997-2025


,companyid,companyname,boardid,boardname,directorid,directorname_emp,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,salary,bonus,totalcompensation,log_totalcompensation,totaldirectcomp,perftotal,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,ceo_turnover,turnover_year
0,422.0,ABB LTD,422.0,ABB LTD,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,894.0,894.0,1787.0,7.488294,1787.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2000
1,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,<NA>,1419.0,7.257708,1419.0,<NA>,507.0,<NA>,384.0,<NA>,384.0,1,2002
2,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,<NA>,2414.0,7.78904,3375.0,<NA>,2707.0,<NA>,727.0,<NA>,727.0,1,2002
3,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,2734.0,648.0,3383.0,8.126518,4479.0,<NA>,3502.0,<NA>,485.0,<NA>,485.0,1,2002
4,422.0,ABB LTD,422.0,ABB LTD,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,993.0,306.0,1299.0,7.16935,1614.0,0.67,24.0,3843.0,<NA>,2599.0,2599.0,1,2005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd Montag,CEO,2018-03-16,NaT,2022-09-01,2022,USD,1353.0,1176.0,2529.0,7.835579,3893.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2018
2065,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd Montag,CEO,2018-03-16,NaT,2023-09-01,2023,USD,1459.0,1515.0,2974.0,7.997663,3676.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2018
2066,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd Montag,CEO,2018-03-16,NaT,2024-09-01,2024,USD,1571.0,1727.0,3297.0,8.100768,4040.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2018
2067,3081662.0,PROSUS NV (Myriad International Holdings NV pr...,3081662.0,PROSUS NV (Myriad International Holdings NV pr...,1045804.0,Fabricio Rocha,Group CEO,2024-08-21,2026-01-01,2025-03-01,2025,USD,542.0,638.0,1180.0,7.07327,1460.0,0.99,5864.0,109729.0,<NA>,109729.0,109729.0,0,2024


Next, we merge in the CEO turnover firm-year indicator created from the employment data. `ceo_turnover_dummy` equals one in years where a new CEO spell starts at the firm. Missing values are set to zero, meaning no observed CEO turnover in that firm-year.

In [516]:
# Merge firm-year CEO turnover indicator into the pay panel
ceo_pay_panel = ceo_pay_panel.merge(
    ceo_turnover_firm_year,
    left_on=["companyid", "annualreportyear"],
    right_on=["companyid", "turnover_year"],
    how="left",
    suffixes=("", "_event")
)

ceo_pay_panel["num_ceo_turnovers"] = ceo_pay_panel["num_ceo_turnovers"].fillna(0).astype(int)
ceo_pay_panel["ceo_turnover_dummy"] = ceo_pay_panel["ceo_turnover_dummy"].fillna(0).astype(int)

# If the merge creates a second turnover-year column, keep it only as event-year information.
if "turnover_year_event" in ceo_pay_panel.columns:
    ceo_pay_panel = ceo_pay_panel.rename(columns={"turnover_year_event": "turnover_event_year"})

ceo_pay_panel[[
    "companyname", "annualreportyear", "directorname_emp", "totalcompensation",
    "ceo_turnover_dummy", "num_ceo_turnovers"
]]

,companyname,annualreportyear,directorname_emp,totalcompensation,ceo_turnover_dummy,num_ceo_turnovers
0,ABB LTD,2001,Jörgen Centerman,1787.0,0,0
1,ABB LTD,2002,Juergen Dormann,1419.0,1,1
2,ABB LTD,2003,Juergen Dormann,2414.0,0,0
3,ABB LTD,2004,Juergen Dormann,3383.0,0,0
4,ABB LTD,2005,Fred Kindle,1299.0,1,1
...,...,...,...,...,...,...
2064,SIEMENS HEALTHINEERS AG,2022,Doctor Bernd Montag,2529.0,0,0
2065,SIEMENS HEALTHINEERS AG,2023,Doctor Bernd Montag,2974.0,0,0
2066,SIEMENS HEALTHINEERS AG,2024,Doctor Bernd Montag,3297.0,0,0
2067,PROSUS NV (Myriad International Holdings NV pr...,2025,Fabricio Rocha,1180.0,0,0


For event-style analysis, each company is assigned its first observed turnover year. `treated_firm` identifies companies that experience at least one observed turnover, while `event_time` measures years relative to that first turnover and `post_turnover` identifies the turnover year and later observations.

In [517]:
first_turnover_year = (
    ceo_turnover_firm_year
    .groupby("companyid")["turnover_year"]
    .min()
    .rename("first_turnover_year")
    .reset_index()
)
print(f"Firms with an observed turnover: {len(first_turnover_year):,}")
print(
    f"Observed first-turnover range: "
    f"{first_turnover_year['first_turnover_year'].min()}-"
    f"{first_turnover_year['first_turnover_year'].max()}"
)
first_turnover_year.head()

Firms with an observed turnover: 414
Observed first-turnover range: 1900-2026


,companyid,first_turnover_year
0,384.0,2012
1,422.0,2002
2,595.0,2016
3,598.0,2006
4,626.0,1983


In [518]:
# Inspect the distribution of first observed turnover years before merging it into the pay panel.
first_turnover_year["first_turnover_year"].describe()

count     414.000000
mean     2007.661836
std        12.190444
min      1900.000000
25%      2000.000000
50%      2008.000000
75%      2017.000000
max      2026.000000
Name: first_turnover_year, dtype: float64

In [519]:
# Create event-time variables around the first observed CEO turnover per firm.
ceo_pay_panel = ceo_pay_panel.merge(
    first_turnover_year,
    on="companyid",
    how="left"
)

ceo_pay_panel["treated_firm"] = ceo_pay_panel["first_turnover_year"].notna().astype(int)
ceo_pay_panel["event_time"] = ceo_pay_panel["annualreportyear"] - ceo_pay_panel["first_turnover_year"]
ceo_pay_panel["post_turnover"] = (
    ceo_pay_panel["treated_firm"].eq(1)
    & ceo_pay_panel["event_time"].ge(0)
).astype(int)

# Useful event-window flags for later robustness checks
ceo_pay_panel["event_window_3yr"] = ceo_pay_panel["event_time"].between(-3, 3).fillna(False).astype(int)
ceo_pay_panel["event_window_5yr"] = ceo_pay_panel["event_time"].between(-5, 5).fillna(False).astype(int)

ceo_pay_panel[[
    "companyname", "annualreportyear", "first_turnover_year", "event_time",
    "treated_firm", "post_turnover", "ceo_turnover_dummy"
]]

,companyname,annualreportyear,first_turnover_year,event_time,treated_firm,post_turnover,ceo_turnover_dummy
0,ABB LTD,2001,2002.0,-1.0,1,0,0
1,ABB LTD,2002,2002.0,0.0,1,1,1
2,ABB LTD,2003,2002.0,1.0,1,1,0
3,ABB LTD,2004,2002.0,2.0,1,1,0
4,ABB LTD,2005,2002.0,3.0,1,1,1
...,...,...,...,...,...,...,...
2064,SIEMENS HEALTHINEERS AG,2022,NaN,NaN,0,0,0
2065,SIEMENS HEALTHINEERS AG,2023,NaN,NaN,0,0,0
2066,SIEMENS HEALTHINEERS AG,2024,NaN,NaN,0,0,0
2067,PROSUS NV (Myriad International Holdings NV pr...,2025,2024.0,1.0,1,1,0


Finally, we create within-firm CEO pay growth variables. These are useful descriptive outcomes because they measure whether CEO compensation changes sharply after leadership transitions.

In [520]:
# Create within-firm CEO pay growth variables
ceo_pay_panel = ceo_pay_panel.sort_values(["companyid", "annualreportyear"]).copy()

ceo_pay_panel["lag_totalcompensation"] = (
    ceo_pay_panel
    .groupby("companyid")["totalcompensation"]
    .shift(1)
)

ceo_pay_panel["lag_log_totalcompensation"] = (
    ceo_pay_panel
    .groupby("companyid")["log_totalcompensation"]
    .shift(1)
)

ceo_pay_panel["pay_change"] = (
    ceo_pay_panel["totalcompensation"]
    - ceo_pay_panel["lag_totalcompensation"]
)

ceo_pay_panel["pay_change_pct"] = (
    ceo_pay_panel["pay_change"]
    / ceo_pay_panel["lag_totalcompensation"]
)

ceo_pay_panel["log_pay_change"] = (
    ceo_pay_panel["log_totalcompensation"]
    - ceo_pay_panel["lag_log_totalcompensation"]
)

ceo_pay_panel[[
    "companyname", "annualreportyear", "totalcompensation", "lag_totalcompensation",
    "pay_change", "pay_change_pct", "log_pay_change", "ceo_turnover_dummy", "post_turnover"
]]

,companyname,annualreportyear,totalcompensation,lag_totalcompensation,pay_change,pay_change_pct,log_pay_change,ceo_turnover_dummy,post_turnover
0,ABB LTD,2001,1787.0,<NA>,<NA>,<NA>,<NA>,0,0
1,ABB LTD,2002,1419.0,1787.0,-368.0,-0.205932,-0.230586,1,1
2,ABB LTD,2003,2414.0,1419.0,995.0,0.701198,0.531333,0,1
3,ABB LTD,2004,3383.0,2414.0,969.0,0.401408,0.337478,0,1
4,ABB LTD,2005,1299.0,3383.0,-2084.0,-0.616021,-0.957168,1,1
...,...,...,...,...,...,...,...,...,...
2064,SIEMENS HEALTHINEERS AG,2022,2529.0,2733.0,-204.0,-0.074643,-0.077576,0,0
2065,SIEMENS HEALTHINEERS AG,2023,2974.0,2529.0,445.0,0.175959,0.162084,0,0
2066,SIEMENS HEALTHINEERS AG,2024,3297.0,2974.0,323.0,0.108608,0.103105,0,0
2067,PROSUS NV (Myriad International Holdings NV pr...,2025,1180.0,<NA>,<NA>,<NA>,<NA>,0,1


<a id="market-data-and-market-capitalization"></a>
## 7. Market Data and Market Capitalization

Annual fundamentals do not contain the monthly security price needed to construct market capitalization. We therefore retrieve Compustat monthly security data for the matched `gvkey` values, select one exchange per firm, and attach the nearest price within 31 days of each fiscal reporting date.


In [521]:
# Retain the annual fundamentals needed for the final analysis panel.
fundamental_cols = [
    "gvkey", "isin", "datadate", "fyear", "at", "sale",
    "ebit", "dltt", "nicon", "emp", "cshoi",
]
df_firm = df_firm[fundamental_cols].copy()
df_firm

,gvkey,isin,datadate,fyear,at,sale,ebit,dltt,nicon,emp,cshoi
0,210835,GB00B1YW4409,2001-03-31,2000,7439.0,<NA>,-53.0,1089.0,-142.0,<NA>,607.437
1,210835,GB00B1YW4409,2002-03-31,2001,6133.0,<NA>,-823.0,1271.0,-960.0,<NA>,599.887
2,210835,GB00B1YW4409,2003-03-31,2002,4999.0,<NA>,-826.0,1071.0,140.0,<NA>,610.918
3,210835,GB00B1YW4409,2004-03-31,2003,5412.0,<NA>,628.0,1423.0,531.0,0.75,603.594
4,210835,GB00B1YW4409,2005-03-31,2004,5701.0,<NA>,597.0,1312.0,512.0,0.74,601.913
...,...,...,...,...,...,...,...,...,...,...,...
12496,220426,CH0011075394,2020-12-31,2020,439299.0,<NA>,5737.0,15341.0,3834.0,52.93,148.496
12497,220426,CH0011075394,2021-12-31,2021,435826.0,<NA>,7974.0,16190.0,5202.0,<NA>,148.291
12498,220426,CH0011075394,2022-12-31,2022,377782.0,<NA>,6684.0,15056.0,4603.0,60.0,147.533
12499,220426,CH0011075394,2023-12-31,2023,361382.0,<NA>,17981.0,13946.0,4351.0,60.0,143.99


In [522]:
start_time = time.time()

# Pull monthly security prices for the matched firms when not cached
gvkey_list = (
    df_firm["gvkey"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
    .tolist()
)

def fetch_compustat_market_data():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(gvkey_list), chunk_size):
        gvkey_chunk = tuple(gvkey_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT gvkey, iid, isin, datadate, conm, prccm, curcdm, secstat, tpci, exchg
            FROM comp.g_secm
            WHERE gvkey IN %(gvkeys)s
            ORDER BY gvkey, datadate
        """, params={"gvkeys": gvkey_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_market = load_or_fetch(
    "compustat_monthly_market_stoxx600",
    fetch_compustat_market_data,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 GVKEYs used for WRDS query: {len(gvkey_list):,}")
print(f"Monthly market records available: {len(df_market):,}")
print(f"Unique GVKEYs matched: {df_market['gvkey'].nunique():,}")

print_running_time(start_time, label="Market data load time")

df_market.head()

Loaded cached data: ../data/wrds_cache/compustat_monthly_market_stoxx600.pkl (394,739 rows)
STOXX 600 GVKEYs used for WRDS query: 556
Monthly market records available: 394,739
Unique GVKEYs matched: 552
Market data load time: 0.19 seconds


,gvkey,iid,isin,datadate,conm,prccm,curcdm,secstat,tpci,exchg
0,001166,02W,<NA>,2007-01-31,ASM INTERNATIONAL NV,17.67,EUR,A,0,171
1,001166,01W,NL0000334118,2007-01-31,ASM INTERNATIONAL NV,17.58,EUR,A,0,104
2,001166,02W,<NA>,2007-02-28,ASM INTERNATIONAL NV,17.56,EUR,A,0,171
3,001166,01W,NL0000334118,2007-02-28,ASM INTERNATIONAL NV,17.43,EUR,A,0,104
4,001166,02W,<NA>,2007-03-31,ASM INTERNATIONAL NV,16.45,EUR,A,0,171


**Table description:** `df_market` contains monthly Compustat security prices. Because the same `gvkey` can be listed on multiple exchanges, this table is checked before choosing one exchange per firm.


In [523]:
missing_values_table(df_market)

,missing_count,missing_percent
isin,263752,66.82
prccm,6416,1.63
gvkey,0,0.00
iid,0,0.00
datadate,0,0.00
conm,0,0.00
curcdm,0,0.00
secstat,4,0.00
tpci,4,0.00
exchg,4,0.00


In [524]:
exchanges = (
    df_market["exchg"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

def fetch_exchange_codes():
    wrds_conn = get_wrds_connection()
    return wrds_conn.raw_sql("""
        SELECT *
        FROM comp_na_daily_all.r_ex_codes
        WHERE exchgcd IN %(exchanges)s
    """, params={"exchanges": tuple(exchanges)})


df_exchanges = load_or_fetch(
    "compustat_exchange_codes",
    fetch_exchange_codes,
    refresh=REFRESH_WRDS_CACHE,
)

df_exchanges

Loaded cached data: ../data/wrds_cache/compustat_exchange_codes.pkl (40 rows)


,exchgcd,exchgdesc
0,329,Kazakhstan Stock Exchange
1,341,Hong Kong-Shanghai Stock Connect (SB)
2,348,Hong Kong-Shenzhen Stock Connect (SB)
3,349,BATS Chi-X Europe
4,171,XETRA
5,172,Irish Stock Exchange All Market
6,177,Johannesburg Stock Exchange
7,185,Nigerian Stock Exchange
8,104,NYSE Euronext Amsterdam
9,106,ASX All Markets


In [525]:
# Merge exchange names into the market data for better interpretability
df_market = df_market.merge(df_exchanges, left_on="exchg", right_on="exchgcd", how="left")
df_market

,gvkey,iid,isin,datadate,conm,prccm,curcdm,secstat,tpci,exchg,exchgcd,exchgdesc
0,001166,02W,<NA>,2007-01-31,ASM INTERNATIONAL NV,17.67,EUR,A,0,171,171,XETRA
1,001166,01W,NL0000334118,2007-01-31,ASM INTERNATIONAL NV,17.58,EUR,A,0,104,104,NYSE Euronext Amsterdam
2,001166,02W,<NA>,2007-02-28,ASM INTERNATIONAL NV,17.56,EUR,A,0,171,171,XETRA
3,001166,01W,NL0000334118,2007-02-28,ASM INTERNATIONAL NV,17.43,EUR,A,0,104,104,NYSE Euronext Amsterdam
4,001166,02W,<NA>,2007-03-31,ASM INTERNATIONAL NV,16.45,EUR,A,0,171,171,XETRA
...,...,...,...,...,...,...,...,...,...,...,...,...
394734,370750,01W,NL0015073TS8,2026-01-31,CSG NV,30.555,EUR,A,0,104,104,NYSE Euronext Amsterdam
394735,370750,01W,NL0015073TS8,2026-02-28,CSG NV,31.83,EUR,A,0,104,104,NYSE Euronext Amsterdam
394736,370750,01W,NL0015073TS8,2026-03-31,CSG NV,23.34,EUR,A,0,104,104,NYSE Euronext Amsterdam
394737,370750,01W,NL0015073TS8,2026-04-30,CSG NV,18.416,EUR,A,0,104,104,NYSE Euronext Amsterdam


### 7.1 Filter Market Data to One Exchange per Firm


**Why this filter matters:** Some companies have monthly prices from several exchanges. To avoid duplicate firm-month observations, `df_market_filtered` keeps the exchange with the most complete non-missing price-date coverage for each `gvkey`.


In [526]:
# Keep one exchange per gvkey: choose the exchange with the most complete price-date coverage.
df_market = df_market.copy()

if "market_datadate" not in df_market.columns:
    df_market["market_datadate"] = pd.to_datetime(df_market["datadate"], errors="coerce")
else:
    df_market["market_datadate"] = pd.to_datetime(df_market["market_datadate"], errors="coerce")

df_market["_priced_market_datadate"] = df_market["market_datadate"].where(
    df_market["prccm"].notna()
)

exchange_coverage = (
    df_market
    .groupby(["gvkey", "exchg", "exchgdesc"], dropna=False)
    .agg(
        price_date_count=("_priced_market_datadate", "nunique"),
        price_obs_count=("prccm", "count"),
        total_rows=("market_datadate", "size"),
        first_market_datadate=("market_datadate", "min"),
        last_market_datadate=("market_datadate", "max"),
    )
    .reset_index()
)

selected_exchanges = (
    exchange_coverage
    .sort_values(
        ["gvkey", "price_date_count", "price_obs_count", "last_market_datadate", "total_rows"],
        ascending=[True, False, False, False, False],
    )
    .drop_duplicates("gvkey")
)

df_market_filtered = (
    df_market
    .merge(
        selected_exchanges[["gvkey", "exchg"]].assign(_selected_exchange=1),
        on=["gvkey", "exchg"],
        how="inner",
    )
    .drop(columns=["_priced_market_datadate", "_selected_exchange"])
    .sort_values(["gvkey", "market_datadate"])
    .reset_index(drop=True)
)

multi_exchange_gvkeys = exchange_coverage.groupby("gvkey")["exchg"].nunique().gt(1).sum()

print(f"Raw market rows:        {len(df_market):,}")
print(f"Filtered market rows:   {len(df_market_filtered):,}")
print(f"GVKEYs retained:        {df_market_filtered['gvkey'].nunique():,}")
print(f"GVKEYs with >1 exchange before filtering: {multi_exchange_gvkeys:,}")

display(selected_exchanges.head(10))
df_market_filtered


Raw market rows:        394,739
Filtered market rows:   129,745
GVKEYs retained:        552
GVKEYs with >1 exchange before filtering: 541


,gvkey,exchg,exchgdesc,price_date_count,price_obs_count,total_rows,first_market_datadate,last_market_datadate
0,001166,104,NYSE Euronext Amsterdam,233,233,233,2007-01-31,2026-05-31
10,001932,194,London Stock Exchange,233,233,233,2007-01-31,2026-05-31
16,002410,194,London Stock Exchange,233,699,699,2007-01-31,2026-05-31
19,002411,194,London Stock Exchange,233,233,233,2007-01-31,2026-05-31
25,004439,256,NASDAQ OMX Nordic,233,466,466,2007-01-31,2026-05-31
29,005180,194,London Stock Exchange,233,233,233,2007-01-31,2026-05-31
32,008546,104,NYSE Euronext Amsterdam,233,269,269,2007-01-31,2026-05-31
38,011217,256,NASDAQ OMX Nordic,233,466,466,2007-01-31,2026-05-31
44,011749,256,NASDAQ OMX Nordic,233,468,468,2007-01-31,2026-05-31
48,012368,256,NASDAQ OMX Nordic,233,466,466,2007-01-31,2026-05-31


,gvkey,iid,isin,datadate,conm,prccm,curcdm,secstat,tpci,exchg,exchgcd,exchgdesc,market_datadate
0,001166,01W,NL0000334118,2007-01-31,ASM INTERNATIONAL NV,17.58,EUR,A,0,104,104,NYSE Euronext Amsterdam,2007-01-31
1,001166,01W,NL0000334118,2007-02-28,ASM INTERNATIONAL NV,17.43,EUR,A,0,104,104,NYSE Euronext Amsterdam,2007-02-28
2,001166,01W,NL0000334118,2007-03-31,ASM INTERNATIONAL NV,16.65,EUR,A,0,104,104,NYSE Euronext Amsterdam,2007-03-31
3,001166,01W,NL0000334118,2007-04-30,ASM INTERNATIONAL NV,18.0,EUR,A,0,104,104,NYSE Euronext Amsterdam,2007-04-30
4,001166,01W,NL0000334118,2007-05-31,ASM INTERNATIONAL NV,19.6,EUR,A,0,104,104,NYSE Euronext Amsterdam,2007-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129740,370750,01W,NL0015073TS8,2026-01-31,CSG NV,30.555,EUR,A,0,104,104,NYSE Euronext Amsterdam,2026-01-31
129741,370750,01W,NL0015073TS8,2026-02-28,CSG NV,31.83,EUR,A,0,104,104,NYSE Euronext Amsterdam,2026-02-28
129742,370750,01W,NL0015073TS8,2026-03-31,CSG NV,23.34,EUR,A,0,104,104,NYSE Euronext Amsterdam,2026-03-31
129743,370750,01W,NL0015073TS8,2026-04-30,CSG NV,18.416,EUR,A,0,104,104,NYSE Euronext Amsterdam,2026-04-30


### 7.2 Merge Fundamentals with Monthly Market Prices


**Table description:** `df_funda` merges annual accounting fundamentals with the nearest monthly market price within 31 days of the fiscal reporting date. This does not average monthly prices; it attaches the closest available market observation to each annual firm record.


In [527]:
# Make sure dates are datetime
df_firm["datadate"] = pd.to_datetime(df_firm["datadate"])
df_market_filtered["market_datadate"] = pd.to_datetime(df_market_filtered["market_datadate"])

# Make sure gvkey has the same type/format in both datasets
df_firm["gvkey"] = df_firm["gvkey"].astype(str).str.strip()
df_market_filtered["gvkey"] = df_market_filtered["gvkey"].astype(str).str.strip()

# IMPORTANT: for merge_asof, sort by date first, then gvkey
df_firm_sorted = df_firm.sort_values(["datadate", "gvkey"]).reset_index(drop=True)
df_market_sorted = df_market_filtered.sort_values(["market_datadate", "gvkey"]).reset_index(drop=True)

df_funda = pd.merge_asof(
    df_firm_sorted,
    df_market_sorted,
    left_on="datadate",
    right_on="market_datadate",
    by="gvkey",
    direction="nearest",
    tolerance=pd.Timedelta("31 days")
)

# Calculate market cap
df_funda["market_cap"] = df_funda["prccm"] * df_funda["cshoi"]
df_funda

,gvkey,isin_x,datadate_x,fyear,at,sale,ebit,dltt,nicon,emp,cshoi,iid,isin_y,datadate_y,conm,prccm,curcdm,secstat,tpci,exchg,exchgcd,exchgdesc,market_datadate,market_cap
0,018636,GB0002374006,2000-06-30,2000,16136.0,11870.0,1977.0,3751.0,976.0,72.474,3422.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>
1,100171,GB0000811801,2000-06-30,2000,1068.3,1250.0,150.8,26.5,100.2,3.188,233.527,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>
2,100953,DE0007037129,2000-06-30,2000,64989.0,42426.0,267.0,1333.0,1212.0,152.132,473.012,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>
3,210652,GB00BYRJ5J14,2000-06-30,2000,52.783,<NA>,2.873,29.5,1.116,0.0,15.7,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>
4,100045,GB00B1WY2338,2000-07-31,2000,1303.3,1463.7,265.6,217.6,177.0,15.521,317.215,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12496,319417,GB00BVYVFW23,2025-03-31,2024,639.6,601.1,373.2,0.4,282.6,1.29,879.805,01W,GB00BVYVFW23,2025-03-31,AUTOTRADER GROUP PLC,7.442,GBP,A,0,194,194,London Stock Exchange,2025-03-31,6547.50881
12497,332739,GB00BKDRYJ47,2025-03-31,2024,12023.0,4955.0,1473.0,4656.0,220.0,4.253,3668.349,01W,GB00BKDRYJ47,2025-03-31,AIRTEL AFRICA PLC,1.656,GBP,A,0,194,194,London Stock Exchange,2025-03-31,6074.785944
12498,333645,NL0013654783,2025-03-31,2024,72588.0,6170.0,188.0,15053.0,12495.0,23.323,2280.205,02W,NL0013654783,2025-03-31,PROSUS NV,42.575,EUR,A,0,104,104,NYSE Euronext Amsterdam,2025-03-31,97079.727875
12499,206248,GB0003718474,2025-05-31,2024,383.0,617.5,262.5,34.0,196.1,3.015,32.972,01W,GB0003718474,2025-05-31,GAMES WORKSHOP GROUP PLC,153.3,GBP,A,0,194,194,London Stock Exchange,2025-05-31,5054.6076


---

<a id="final-analysis-panel"></a>
## 8. Final Analysis Panel

`ceo_pay_panel` is the firm-year backbone. Fundamentals and market capitalization are attached by `isin` and `annualreportyear`; executive age and gender are attached by `directorid`; and sector and headquarters country are attached through a de-duplicated company bridge. The merge is designed to preserve exactly one observation per company-year.


### 8.1 Construct the Analysis-Ready Panel


In [528]:
FUNDA_COLS = [
    "gvkey", "isin_x", "datadate_x", "fyear", "at", "sale",
    "ebit", "dltt", "nicon", "emp", "cshoi", "market_cap",
]
EXEC_COLS = ["directorid", "age", "gender"]
CEO_PAY_COLS = list(ceo_pay_panel.columns)


In [529]:
# Merge CEO pay, firm fundamentals, market cap, and executive demographics into one final panel.
funda_cols_available = [col for col in FUNDA_COLS if col in df_funda.columns]
exec_cols_available = [col for col in EXEC_COLS if col in df_exec.columns]

ceo_panel_for_merge = ceo_pay_panel[list(CEO_PAY_COLS)].copy()

if "isin" not in ceo_panel_for_merge.columns:
    firm_isin_bridge = (
        df_emp[["companyid", "isin"]]
        .dropna(subset=["companyid", "isin"])
        .drop_duplicates(subset=["companyid"])
    )
    ceo_panel_for_merge = ceo_panel_for_merge.merge(
        firm_isin_bridge,
        on="companyid",
        how="left",
    )

df_funda_final = (
    df_funda[funda_cols_available]
    .rename(columns={
        "isin_x": "isin",
        "datadate_x": "datadate",
        "fyear": "annualreportyear",
    })
)

if "isin" not in ceo_panel_for_merge.columns or "isin" not in df_funda_final.columns:
    raise KeyError("No ISIN identifier available to merge ceo_pay_panel with df_funda_final.")

merge_keys = ["isin", "annualreportyear"]

df_funda_final = df_funda_final.drop_duplicates(subset=merge_keys)

df_exec_final = (
    df_exec[exec_cols_available]
    .drop_duplicates(subset=["directorid"])
)

# Build a one-row-per-company bridge for sector and headquarters country.
# Prefer the most complete context row when a company has several employment records.
firm_context = (
    df_emp[["companyid", "sector", "hocountryname"]]
    .dropna(subset=["companyid"])
    .assign(
        _missing_context=lambda x: x[["sector", "hocountryname"]]
        .isna()
        .sum(axis=1)
    )
    .sort_values(["companyid", "_missing_context"])
    .drop_duplicates(subset=["companyid"], keep="first")
    .drop(columns="_missing_context")
)

df_panel_final = (
    ceo_panel_for_merge
    .merge(df_funda_final, on=merge_keys, how="left")
    .merge(df_exec_final, on="directorid", how="left")
    .merge(firm_context, on="companyid", how="left")
)

print(f"Merge keys:              {merge_keys}")
print(f"CEO pay panel rows:      {len(ceo_pay_panel):,}")
print(f"Final panel rows:        {len(df_panel_final):,}")
print(f"Final panel columns:     {df_panel_final.shape[1]:,}")
print(f"Matched fundamentals:    {df_panel_final['gvkey'].notna().sum():,}")
print(f"Matched executive data:  {df_panel_final['age'].notna().sum():,}")
print(f"Matched sector data:     {df_panel_final['sector'].notna().sum():,}")

df_panel_final.head()


Merge keys:              ['isin', 'annualreportyear']
CEO pay panel rows:      2,069
Final panel rows:        2,069
Final panel columns:     54
Matched fundamentals:    2,002
Matched executive data:  2,065
Matched sector data:     2,069


,companyid,companyname,boardid,boardname,directorid,directorname_emp,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,salary,bonus,totalcompensation,log_totalcompensation,totaldirectcomp,perftotal,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,ceo_turnover,turnover_year,turnover_event_year,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,lag_totalcompensation,lag_log_totalcompensation,pay_change,pay_change_pct,log_pay_change,isin,gvkey,datadate,at,sale,ebit,dltt,nicon,emp,cshoi,market_cap,age,gender,sector,hocountryname
0,422.0,ABB LTD,422.0,ABB LTD,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,894.0,894.0,1787.0,7.488294,1787.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,CH0012221716,210418,2001-12-31,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,<NA>,77,M,Engineering & Machinery,Switzerland
1,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,<NA>,1419.0,7.257708,1419.0,<NA>,507.0,<NA>,384.0,<NA>,384.0,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,1787.0,7.488294,-368.0,-0.205932,-0.230586,CH0012221716,210418,2002-12-31,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,<NA>,86,M,Engineering & Machinery,Switzerland
2,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,<NA>,2414.0,7.78904,3375.0,<NA>,2707.0,<NA>,727.0,<NA>,727.0,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,1419.0,7.257708,995.0,0.701198,0.531333,CH0012221716,210418,2003-12-31,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,<NA>,86,M,Engineering & Machinery,Switzerland
3,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,2734.0,648.0,3383.0,8.126518,4479.0,<NA>,3502.0,<NA>,485.0,<NA>,485.0,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,2414.0,7.78904,969.0,0.401408,0.337478,CH0012221716,210418,2004-12-31,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,<NA>,86,M,Engineering & Machinery,Switzerland
4,422.0,ABB LTD,422.0,ABB LTD,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,993.0,306.0,1299.0,7.16935,1614.0,0.67,24.0,3843.0,<NA>,2599.0,2599.0,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,3383.0,8.126518,-2084.0,-0.616021,-0.957168,CH0012221716,210418,2005-12-31,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,<NA>,67,M,Engineering & Machinery,Switzerland


In [530]:
# Derive financial ratios needed as covariates
df_panel_final["roa"] = df_panel_final["nicon"] / df_panel_final["at"].replace(0, np.nan)
df_panel_final["ebit_margin"] = df_panel_final["ebit"] / df_panel_final["sale"].replace(0, np.nan)
df_panel_final["leverage"] = df_panel_final["dltt"] / df_panel_final["at"].replace(0, np.nan)
df_panel_final["log_assets"] = np.log(df_panel_final["at"].replace(0, np.nan))

df_panel_final[["isin", "annualreportyear", "roa", "ebit_margin", "leverage", "log_assets"]].head()


,isin,annualreportyear,roa,ebit_margin,leverage,log_assets
0,CH0012221716,2001,-0.004019,0.019641,0.155918,10.384184
1,CH0012221716,2002,0.003284,0.032687,0.182034,10.293264
2,CH0012221716,2003,0.002828,0.034903,0.206819,10.322625
3,CH0012221716,2004,0.018155,0.052459,0.198606,10.113627
4,CH0012221716,2005,0.039639,0.080118,0.176558,10.011265


In [531]:
df_panel_final

,companyid,companyname,boardid,boardname,directorid,directorname_emp,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,salary,bonus,totalcompensation,log_totalcompensation,totaldirectcomp,perftotal,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,ceo_turnover,turnover_year,turnover_event_year,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,lag_totalcompensation,lag_log_totalcompensation,pay_change,pay_change_pct,log_pay_change,isin,gvkey,datadate,at,sale,ebit,dltt,nicon,emp,cshoi,market_cap,age,gender,sector,hocountryname,roa,ebit_margin,leverage,log_assets
0,422.0,ABB LTD,422.0,ABB LTD,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,894.0,894.0,1787.0,7.488294,1787.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,CH0012221716,210418,2001-12-31,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,<NA>,77,M,Engineering & Machinery,Switzerland,-0.004019,0.019641,0.155918,10.384184
1,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,<NA>,1419.0,7.257708,1419.0,<NA>,507.0,<NA>,384.0,<NA>,384.0,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,1787.0,7.488294,-368.0,-0.205932,-0.230586,CH0012221716,210418,2002-12-31,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,<NA>,86,M,Engineering & Machinery,Switzerland,0.003284,0.032687,0.182034,10.293264
2,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,<NA>,2414.0,7.78904,3375.0,<NA>,2707.0,<NA>,727.0,<NA>,727.0,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,1419.0,7.257708,995.0,0.701198,0.531333,CH0012221716,210418,2003-12-31,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,<NA>,86,M,Engineering & Machinery,Switzerland,0.002828,0.034903,0.206819,10.322625
3,422.0,ABB LTD,422.0,ABB LTD,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,2734.0,648.0,3383.0,8.126518,4479.0,<NA>,3502.0,<NA>,485.0,<NA>,485.0,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,2414.0,7.78904,969.0,0.401408,0.337478,CH0012221716,210418,2004-12-31,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,<NA>,86,M,Engineering & Machinery,Switzerland,0.018155,0.052459,0.198606,10.113627
4,422.0,ABB LTD,422.0,ABB LTD,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,993.0,306.0,1299.0,7.16935,1614.0,0.67,24.0,3843.0,<NA>,2599.0,2599.0,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,3383.0,8.126518,-2084.0,-0.616021,-0.957168,CH0012221716,210418,2005-12-31,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,<NA>,67,M,Engineering & Machinery,Switzerland,0.039639,0.080118,0.176558,10.011265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd Montag,CEO,2018-03-16,NaT,2022-09-01,2022,USD,1353.0,1176.0,2529.0,7.835579,3893.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2733.0,7.913155,-204.0,-0.074643,-0.077576,DE000SHL1006,326765,2022-09-30,49056.0,21714.0,2924.0,13811.0,2038.0,69.5,1119.394,44335.838158,56,M,Health,Germany,0.041544,0.13466,0.281535,10.800718
2065,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd Montag,CEO,2018-03-16,NaT,2023-09-01,2023,USD,1459.0,1515.0,2974.0,7.997663,3676.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2529.0,7.835579,445.0,0.175959,0.162084,DE000SHL1006,326765,2023-09-30,46684.0,21680.0,2133.0,12258.0,1509.0,71.0,1115.788,47957.684028,56,M,Health,Germany,0.032324,0.098386,0.262574,10.751157
2066,2794683.0,SIEMENS HEALTHINEERS AG,2794683.0,SIEMENS HEALTHINEERS AG,831291.0,Doctor Bernd

In [532]:
# Keep identifiers, treatment timing, outcomes, and auditable firm controls.
final_cols = [
    "isin", "gvkey", "companyid", "companyname", "sector", "hocountryname",
    "directorid", "directorname_emp", "gender", "age", "rolename_emp",
    "datestartrole", "dateendrole_clean", "annualreportdate", "annualreportyear",
    "currency", "totalcompensation", "log_totalcompensation",
    "ceo_turnover", "turnover_year", "turnover_event_year", "num_ceo_turnovers",
    "ceo_turnover_dummy", "first_turnover_year", "treated_firm", "event_time",
    "post_turnover", "event_window_3yr", "event_window_5yr",
    "lag_totalcompensation", "lag_log_totalcompensation", "pay_change",
    "pay_change_pct", "log_pay_change",
    "at", "sale", "ebit", "dltt", "nicon", "emp", "cshoi", "market_cap",
    "roa", "ebit_margin", "leverage", "log_assets",
]

missing_final_cols = [col for col in final_cols if col not in df_panel_final.columns]
if missing_final_cols:
    raise KeyError(f"Expected final-panel columns are missing: {missing_final_cols}")

df_panel_final = df_panel_final[final_cols].copy()
df_panel_final

,isin,gvkey,companyid,companyname,sector,hocountryname,directorid,directorname_emp,gender,age,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,log_totalcompensation,ceo_turnover,turnover_year,turnover_event_year,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,lag_totalcompensation,lag_log_totalcompensation,pay_change,pay_change_pct,log_pay_change,at,sale,ebit,dltt,nicon,emp,cshoi,market_cap,roa,ebit_margin,leverage,log_assets
0,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,14884.0,Jörgen Centerman,M,77,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,7.488294,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,<NA>,-0.004019,0.019641,0.155918,10.384184
1,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,7.257708,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,1787.0,7.488294,-368.0,-0.205932,-0.230586,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,<NA>,0.003284,0.032687,0.182034,10.293264
2,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,7.78904,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,1419.0,7.257708,995.0,0.701198,0.531333,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,<NA>,0.002828,0.034903,0.206819,10.322625
3,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,8.126518,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,2414.0,7.78904,969.0,0.401408,0.337478,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,<NA>,0.018155,0.052459,0.198606,10.113627
4,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,4611.0,Fred Kindle,M,67,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,7.16935,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,3383.0,8.126518,-2084.0,-0.616021,-0.957168,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,<NA>,0.039639,0.080118,0.176558,10.011265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2022-09-01,2022,USD,2529.0,7.835579,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2733.0,7.913155,-204.0,-0.074643,-0.077576,49056.0,21714.0,2924.0,13811.0,2038.0,69.5,1119.394,44335.838158,0.041544,0.13466,0.281535,10.800718
2065,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2023-09-01,2023,USD,2974.0,7.997663,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2529.0,7.835579,445.0,0.175959,0.162084,46684.0,21680.0,2133.0,12258.0,1509.0,71.0,1115.788,47957.684028,0.032324,0.098386,0.262574,10.751157
2066,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2024-09-01,2024,USD,3297.0,8.100768,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2974.0,7.997663,323.0,0.108608,0.103105,46055.0,22363.0,2809.0,13455.0,1942.0,72.0,1119.27,56574.62142,0.042167,0.125609,0.292151,10.737592
2067,NL0013654783,NaN,3081662.0,PROSUS NV (Myriad International Holdings NV pr...,Speciality & Other Finance,Netherlands,1045804.0,Fabricio Rocha,M,49,Group CEO,2024-08-21,2026-01-01,2025-03-01,2025,USD,1180.0,7.07327,0,2024,NaN,0,0,2024.0,1,1.0,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [533]:
# Confirm that the final table still has exactly one observation per firm-year.
duplicate_firm_years = df_panel_final.duplicated(
    subset=["companyid", "annualreportyear"],
    keep=False,
).sum()

print(f"Final panel shape: {df_panel_final.shape}")
print(f"Unique companies: {df_panel_final['companyid'].nunique():,}")
print(f"Duplicate firm-year rows: {duplicate_firm_years:,}")

if duplicate_firm_years:
    raise ValueError("The final panel is no longer unique by company-year.")

df_panel_final = df_panel_final.sort_values(
    ["companyid", "annualreportyear"]
).reset_index(drop=True)
df_panel_final

Final panel shape: (2069, 46)
Unique companies: 138
Duplicate firm-year rows: 0


,isin,gvkey,companyid,companyname,sector,hocountryname,directorid,directorname_emp,gender,age,rolename_emp,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,log_totalcompensation,ceo_turnover,turnover_year,turnover_event_year,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,lag_totalcompensation,lag_log_totalcompensation,pay_change,pay_change_pct,log_pay_change,at,sale,ebit,dltt,nicon,emp,cshoi,market_cap,roa,ebit_margin,leverage,log_assets
0,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,14884.0,Jörgen Centerman,M,77,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,7.488294,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,<NA>,-0.004019,0.019641,0.155918,10.384184
1,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,7.257708,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,1787.0,7.488294,-368.0,-0.205932,-0.230586,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,<NA>,0.003284,0.032687,0.182034,10.293264
2,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,7.78904,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,1419.0,7.257708,995.0,0.701198,0.531333,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,<NA>,0.002828,0.034903,0.206819,10.322625
3,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,11152.0,Juergen Dormann,M,86,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,8.126518,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,2414.0,7.78904,969.0,0.401408,0.337478,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,<NA>,0.018155,0.052459,0.198606,10.113627
4,CH0012221716,210418,422.0,ABB LTD,Engineering & Machinery,Switzerland,4611.0,Fred Kindle,M,67,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,7.16935,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,3383.0,8.126518,-2084.0,-0.616021,-0.957168,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,<NA>,0.039639,0.080118,0.176558,10.011265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2022-09-01,2022,USD,2529.0,7.835579,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2733.0,7.913155,-204.0,-0.074643,-0.077576,49056.0,21714.0,2924.0,13811.0,2038.0,69.5,1119.394,44335.838158,0.041544,0.13466,0.281535,10.800718
2065,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2023-09-01,2023,USD,2974.0,7.997663,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2529.0,7.835579,445.0,0.175959,0.162084,46684.0,21680.0,2133.0,12258.0,1509.0,71.0,1115.788,47957.684028,0.032324,0.098386,0.262574,10.751157
2066,DE000SHL1006,326765,2794683.0,SIEMENS HEALTHINEERS AG,Health,Germany,831291.0,Doctor Bernd Montag,M,56,CEO,2018-03-16,NaT,2024-09-01,2024,USD,3297.0,8.100768,0,2018,NaN,0,0,NaN,0,NaN,0,0,0,2974.0,7.997663,323.0,0.108608,0.103105,46055.0,22363.0,2809.0,13455.0,1942.0,72.0,1119.27,56574.62142,0.042167,0.125609,0.292151,10.737592
2067,NL0013654783,NaN,3081662.0,PROSUS NV (Myriad International Holdings NV pr...,Speciality & Other Finance,Netherlands,1045804.0,Fabricio Rocha,M,49,Group CEO,2024-08-21,2026-01-01,2025-03-01,2025,USD,1180.0,7.07327,0,2024,NaN,0,0,2024.0,1,1.0,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [537]:
missing_values_table(df_panel_final)

,missing_count,missing_percent
turnover_event_year,1820,87.97
dateendrole_clean,528,25.52
market_cap,431,20.83
ebit_margin,388,18.75
sale,388,18.75
roa,161,7.78
nicon,161,7.78
log_pay_change,138,6.67
lag_totalcompensation,138,6.67
lag_log_totalcompensation,138,6.67


---

<a id="planned-methods"></a>
## 9. Planned Methods


The completed preparation pipeline produces `df_panel_final`. The next stage applies one method from each course block, with a shared emphasis on transparent preprocessing and out-of-sample or robustness evaluation.

### 9.1 Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [X] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:* We will model CEO turnover as a governance shock in a DAG and use backdoor adjustment to control for firm fundamentals that affect both turnover and pay. We will estimate the causal effect of turnover using pre/post (event‑study style) comparisons around the turnover year.

### 9.2 Supervised Learning
- [X] Linear / Ridge / Lasso regression
- [X] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [X] Gradient Boosting (XGBoost / LightGBM / sklearn GBM)
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:* Supervised models will benchmark expected compensation conditional on firm and executive characteristics. Linear models provide interpretable baselines and covariate effects, while tree-based and boosting models capture nonlinearities/interactions for more accurate counterfactual pay predictions. We will use Optuna to tune boosting hyperparameters (e.g., depth, learning rate, subsampling) to avoid overfitting and compare against simpler baselines.

### 9.3 Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [ ] Hierarchical clustering
- [X] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* Clustering will segment firms into comparable peer groups before causal estimation and highlight heterogeneous effects across turnover regimes. A VAE will learn low-dimensional representations of firm/executive profiles to detect anomalous pay structures and support exploratory subgroup analysis.

<a id="evaluation-strategy"></a>
## 10. Evaluation Strategy


The mission succeeds if the final notebook produces a transparent, reproducible panel and uses it to answer the research question from three angles:

- **Causal inference:** estimate whether CEO turnover is associated with changes in CEO pay after adjusting for firm characteristics.
- **Prediction:** benchmark compensation models using RMSE/MAE for pay levels and classification metrics where turnover or high-pay outcomes are modeled.
- **Unsupervised learning:** identify firm or executive clusters that reveal heterogeneous compensation patterns.
- **Robustness:** report missing-data coverage, sensitivity to winsorization, and whether results change across event windows.


<a id="work-plan"></a>
## 11. Work Plan


| Step | Owner | Description | Output | Status |
|---:|---|---|---|---|
| 1 | Achmad | Define the STOXX 600 universe and retrieve BoardEx/Compustat inputs | Raw source tables | Complete |
| 2 | Kajetan | Clean identifiers, dates, compensation variables, and missing values | Harmonized source tables | Complete |
| 3 | Achmad | Identify firm-level CEO spells and turnover events | `ceo_turnover_firm_year` | Complete |
| 4 | Kajetan | Match CEO remuneration to valid CEO spells | `ceo_pay_panel` | Complete |
| 5 | Achmad | Add market capitalization and financial ratios | `df_funda` and ratio covariates | Complete |
| 6 | Achmad + Kajetan | Merge the analysis inputs and validate firm-year uniqueness | `df_panel_final` | Complete |
| 7 | Kajetan | Implement causal and event-window analysis | Estimates and robustness checks | Planned |
| 8 | Achmad | Implement supervised and unsupervised learning | Model metrics, clusters, and heterogeneity | Planned |
| 9 | Achmad + Kajetan | Synthesize findings and limitations | Final discussion and conclusion | Planned |


<a id="results-and-discussion"></a>
## 12. Results and Discussion


### 12.1 Causal Inference

The result sections below are intentionally separated by method block. Each block should use `df_panel_final` as the starting dataset, then create any method-specific filtered samples or feature matrices.


In [534]:
# Causal inference analysis

### 12.2 Supervised Learning

In [535]:
# Supervised learning analysis

### 12.3 Unsupervised / Generative Learning

In [536]:
# Unsupervised / generative analysis

### 12.4 Discussion & Conclusion *(complete for final submission)*


*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
